# PEN · Derivações animadas com Manim

Este notebook gera vídeos que **montam as equações e derivações** do modelo de lançamento
(`atmosfera.ipynb`, `forcas_foguete.ipynb`, `lancamento.ipynb`), passo a passo.

Cenas:
1. Gravidade efetiva
2. Temperatura por camadas
3. Densidade — a EDO (hidrostática + gás ideal)
4. Densidade — solução analítica por camadas (frações parciais + cascata de K)
5. Som, Mach e arrasto
6. Empuxo do motor (câmara estrangulada + expansão isentrópica)
7. Separação de estágios (momentum + energia)
8. Integração trapezoidal (Heun)

Cada célula `%%manim` renderiza um vídeo. Troque `-qm` por `-qh` para 1080p60.

In [ ]:
# Instalação (descomente se necessário):
# !pip install manim

  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/701.3 kB ? eta -:--:--
   -------------- ------------------------- 262.1/701.3 kB ? eta -:--:--
   ----------------------------- ---------- 524.3/701.3 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 701.3/701.3 kB 1.1 MB/s  0:00:00
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ----------- ---------------------------- 1.8/6.4 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------  6.3/6.4 MB 15.6 MB/s eta 0:00:01
   -

In [2]:
from manim import *
from manim.utils.ipython_magic import ManimMagic

# Registra o magic %%manim no Jupyter (idempotente).
try:
    _ip = get_ipython()
except NameError:
    _ip = None
if _ip is not None:
    _ip.register_magics(ManimMagic)

config.media_width = "75%"
config.verbosity = "WARNING"
config.background_color = "#000000"

import numpy as np
print("Manim", __version__ if "__version__" in dir() else "ok")

Manim ok


In [3]:
# Célula 3 (versão importada) — física vem direto dos notebooks originais.
import io, contextlib
import numpy as np
import matplotlib
matplotlib.use("Agg")  # evita abrir figuras durante o import dos notebooks

# Os arquivos atmosfera.ipynb / forcas_foguete.ipynb precisam estar
# na MESMA pasta deste notebook (ou no sys.path).
with contextlib.redirect_stdout(io.StringIO()):
    import import_ipynb  # pip install import_ipynb
    from atmosfera import (
        R, g0, T0, Mm, Rg, rho0,
        alturas, beta, K,
        gravidade_efetiva, temperatura, rho,
    )

# Apelidos para os nomes usados nas cenas do Manim
G0, MM, RG, RHO0 = g0, Mm, Rg, rho0
HL = np.array(alturas, float)
BETA = np.array(beta, float)
grav = gravidade_efetiva
temp = temperatura

print("K0 =", K[0], "| rho(11km) =", rho(11000))

K0 = -52.487466456953086 | rho(11km) = 0.3649157471824279


<string>:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [4]:
# Paleta e helpers de "quadro-negro": empilha equações e troca legendas.
INK="#e9eef8"; MUT="#93a1ba"; CYAN="#22d3ee"; PURPLE="#a06bff"
AMBER="#ffb02e"; GREEN="#3ddc84"; RED="#ff4d6d"

class Board:
    def __init__(self, top=3.25, left=-6.6, max_width=13.2, gap=0.30):
        self.y=top; self.left=left; self.max_width=max_width; self.gap=gap
        self.lines=VGroup()
    def line(self, tex, color=INK, font=38):
        m=MathTex(tex, font_size=font, color=color)
        if m.width>self.max_width: m.scale(self.max_width/m.width)
        m.move_to([self.left+m.width/2, self.y-m.height/2, 0])
        self.y -= m.height + self.gap
        self.lines.add(m)
        return m

def cap(text, color=MUT):
    t=Text(text, font_size=22, color=color, line_spacing=0.8)
    if t.width>13.2: t.scale(13.2/t.width)
    t.to_edge(DOWN, buff=0.35)
    return t

def title(scene, txt):
    t=Text(txt, font_size=32, color=CYAN, weight="BOLD").to_edge(UP, buff=0.28)
    scene.play(Write(t), run_time=0.8)
    return t

def step(scene, b, c, tex, text, color=INK, font=38, box=False, wait=0.9):
    eq=b.line(tex, color=color, font=font)
    scene.play(Write(eq), run_time=1.3)
    nc=cap(text)
    scene.play(FadeIn(nc) if c is None else FadeOut(c), run_time=0.4) if c is None else scene.play(FadeOut(c), FadeIn(nc), run_time=0.4)
    scene.wait(wait)
    bx=None
    if box:
        bx=SurroundingRectangle(eq, color=AMBER, buff=0.16)
        scene.play(Create(bx), run_time=0.7)
    return nc, eq, bx

def clear(scene, b):
    scene.play(FadeOut(b.lines), run_time=0.6)
    return Board()

In [27]:
%%manim -qm -v WARNING CenaGravidade
class CenaGravidade(Scene):
    def construct(self):
        title(self, "1 · Gravidade efetiva")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"F = G\,\frac{M_T\, m}{r^2}",
            "Lei da gravitação universal: a força cai com o quadrado da distância ao centro da Terra.")
        c,_,_ = step(self,b,c, r"g(h)=\frac{F}{m}=\frac{G M_T}{(R+h)^2}",
            "Dividindo pela massa de prova, obtemos a aceleração a uma altitude h (r = R + h).")
        c,_,_ = step(self,b,c, r"g_0=\frac{G M_T}{R^2}",
            "No solo (h = 0) isso define g0 = 9,80665 m/s².")
        c,_,_ = step(self,b,c, r"\frac{g(h)}{g_0}=\frac{R^2}{(R+h)^2}",
            "Dividindo as duas expressões, G e M_T desaparecem — só resta a geometria.")
        c,eq,bx = step(self,b,c, r"g(h)=g_0\left(\frac{R}{R+h}\right)^2",
            "Resultado do atmosfera.ipynb, com R = 6 356 766 m (raio polar).", color=CYAN, box=True)
        self.wait(1.0)
        self.play(FadeOut(b.lines), FadeOut(bx), FadeOut(c))
        ax = Axes(
            x_range=[0, 40000, 10000],
            y_range=[0, 10, 2],
            x_length=7,
            y_length=3.4,
            tips=False,
            axis_config={
                "color": MUT,
                "font_size": 20,
                "include_numbers": True,
            },
            x_axis_config={"numbers_to_exclude": [0]},
        ).shift(DOWN * 0.25)
        lab = ax.get_axis_labels(
            MathTex(r"h\ (\text{km})", font_size=26),
            MathTex(r"g\ (\text{m/s}^2)", font_size=26),
        )
        lab=ax.get_axis_labels(MathTex("h"), MathTex("g"))
        curve=ax.plot(lambda x: grav(x*1000), x_range=[0,40000], color=PURPLE)
        dot=Dot(ax.coords_to_point(400, grav(400000)), color=AMBER)
        self.play(Create(ax), Write(lab)); self.play(Create(curve))
        self.play(FadeIn(dot))
        c=cap("A 400 km (ISS), g ≈ 8,7 m/s². Astronautas flutuam por queda livre, não por g = 0.")
        self.play(Write(c)); self.wait(2)

Manim Community v0.22.1

In [28]:
%%manim -qm -v WARNING CenaTemperatura
import numpy as np
from pathlib import Path

# ---------- camadas e temperaturas de base ----------
NOMES = ["Troposfera", "Tropopausa", "Estratosfera I", "Estratosfera II",
         "Estratopausa", "Mesosfera I", "Mesosfera II", "Mesopausa+"]
TB = [T0]
for i in range(len(HL) - 1):
    TB.append(TB[i] + BETA[i] * (HL[i + 1] - HL[i]))

# ---------- rampa térmica: azul (frio) -> vermelho (quente) ----------
def _h2rgb(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)]) / 255.0

def _rgb2h(c):
    c = np.clip(np.round(np.array(c) * 255), 0, 255).astype(int)
    return "#%02x%02x%02x" % tuple(c)

_STOPS = [(0.00, "#1d4ed8"), (0.30, "#22d3ee"), (0.55, "#a3e635"),
          (0.78, "#fbbf24"), (1.00, "#ef4444")]

def cor_T(T, tmin=180.0, tmax=290.0):
    t = float(np.clip((T - tmin) / (tmax - tmin), 0, 1))
    for (a, ca), (b, cb) in zip(_STOPS[:-1], _STOPS[1:]):
        if t <= b:
            u = 0.0 if b == a else (t - a) / (b - a)
            return _rgb2h(_h2rgb(ca) * (1 - u) + _h2rgb(cb) * u)
    return _STOPS[-1][1]

# ---------- asset da Terra ----------
TERRA_PATH = Path("media/assets/terra.png")
FATOR_TERRA = 1.0  # aumente se o PNG tiver margem transparente (ex.: 1.15)


class CenaTemperatura(Scene):
    def construct(self):
        title(self, "2 · Temperatura por camadas")
        b = Board(); c = None

        # ================= PARTE A — derivação =================
        c, _, _ = step(self, b, c, r"\frac{dT}{dh}=\beta(h)",
            "O modelo define um gradiente β constante por camada (US Standard Atmosphere simplificada).")
        c, _, _ = step(self, b, c, r"T(h)=T_0+\int_0^h \beta(z)\,dz",
            "Integrando o gradiente desde o solo, recuperamos T em qualquer altitude.")
        c, eq, bx = step(self, b, c,
            r"T(h)=T_0+\sum_{j=0}^{i-1}\beta_j\,(h_{j+1}-h_j)+\beta_i\,(h-h_i)",
            "β constante por camada ⇒ T é uma reta em cada trecho: o gráfico é uma polilinha.",
            color=CYAN, box=True, font=36)
        self.wait(1.0)
        self.play(FadeOut(b.lines), FadeOut(bx), FadeOut(c))

        # ================= PARTE B — gráfico com trechos RETOS =================
        ax = Axes(x_range=[0, 100, 100], y_range=[150, 300, 50],
                  x_length=10.6, y_length=3.2, tips=False,
                  axis_config={"color": MUT, "stroke_width": 2}).shift(DOWN * 0.75)

        X = lambda km: ax.coords_to_point(km, 150)[0]
        ybot = ax.coords_to_point(0, 150)[1]
        ytop = ax.coords_to_point(0, 300)[1]

        vlines = VGroup(*[
            DashedLine([X(h / 1000), ybot, 0], [X(h / 1000), ytop + 0.14, 0],
                       dash_length=0.09, stroke_width=2,
                       color=PURPLE, stroke_opacity=0.8)
            for h in HL[1:]
        ])

        edges = [0.0] + [h / 1000 for h in HL[1:]] + [100.0]
        nomes = VGroup()
        for i, nm in enumerate(NOMES):
            xc = (X(edges[i]) + X(edges[i + 1])) / 2
            t = Text(nm, font_size=15, color=INK).rotate(PI / 2)
            t.move_to([xc, ytop + 0.24 + t.height / 2, 0])
            nomes.add(t)

        kmlab = VGroup()
        for h in list(HL[1:]) + [0, 100000]:
            t = Text(str(int(h / 1000)), font_size=14, color=MUT)
            t.move_to([X(h / 1000), ybot - 0.28, 0])
            kmlab.add(t)

        ylabs = VGroup(*[
            Text(str(v), font_size=14, color=MUT).move_to([X(0) - 0.35, ax.coords_to_point(0, v)[1], 0])
            for v in (150, 200, 250, 300)
        ])
        unit_x = Text("h (km)", font_size=15, color=MUT).move_to([X(100) + 0.55, ybot - 0.28, 0])
        unit_y = Text("T (K)", font_size=15, color=MUT).move_to([X(0) - 0.45, ytop + 0.30, 0])

        # >>> polilinha exata: um vértice por fronteira, sem suavização <<<
        verts = [ax.coords_to_point(x, temp(x * 1000)) for x in edges]
        curva = VMobject(stroke_color=AMBER, stroke_width=4)
        curva.set_points_as_corners(verts)

        dots = VGroup(*[
            Dot(ax.coords_to_point(h / 1000, temp(h)), color=CYAN, radius=0.055)
            for h in HL[1:]
        ])

        self.play(Create(ax), Write(ylabs), Write(unit_x), Write(unit_y), run_time=1.2)
        self.play(LaggedStart(*[Create(l) for l in vlines], lag_ratio=0.12), run_time=1.4)
        self.play(Write(kmlab), run_time=0.8)
        self.play(LaggedStart(*[Write(n) for n in nomes], lag_ratio=0.10), run_time=1.8)
        self.play(Create(curva), run_time=2.0)
        self.play(FadeIn(dots))
        c = cap("Reta em cada camada, com quebra exata na fronteira: a inclinação de cada trecho é o β daquela camada.")
        self.play(Write(c)); self.wait(2.0)

        grafico = VGroup(ax, vlines, nomes, kmlab, ylabs, unit_x, unit_y, curva, dots)
        self.play(FadeOut(grafico), FadeOut(c))

        # ================= PARTE C — Terra (PNG) + zoom na atmosfera =================
        ec = np.array([-4.9, -0.35, 0.0]); R_E = 1.45

        if TERRA_PATH.exists():
            terra = ImageMobject(str(TERRA_PATH))
            terra.scale_to_fit_width(2 * R_E * FATOR_TERRA)
            terra.move_to(ec)
        else:
            # fallback: só entra se o arquivo não existir, para a cena nunca quebrar
            base = Circle(radius=R_E, stroke_color="#3f8cff", stroke_width=2,
                          fill_color="#0b2a5b", fill_opacity=1).move_to(ec)
            land = VGroup(*[
                Ellipse(width=w, height=h, stroke_width=0,
                        fill_color="#2f7d4f", fill_opacity=0.85).move_to(ec + np.array([dx, dy, 0]))
                for dx, dy, w, h in [(-0.42, 0.48, 0.70, 0.42), (0.34, 0.05, 0.52, 0.80),
                                     (-0.12, -0.62, 0.88, 0.36)]
            ])
            terra = VGroup(base, land)

        n_r, dr = 14, 0.26 / 14
        aneis = VGroup(*[
            Annulus(inner_radius=R_E + i * dr, outer_radius=R_E + (i + 1) * dr,
                    stroke_width=0, fill_opacity=0.92,
                    fill_color=cor_T(temp((i + 0.5) / n_r * 100000))).move_to(ec)
            for i in range(n_r)
        ])
        terra_lab = Text("Terra · R = 6 371 km", font_size=16, color=MUT).move_to(ec + DOWN * (R_E + 0.55))

        self.play(FadeIn(terra), run_time=1.0)
        self.play(LaggedStart(*[FadeIn(a) for a in aneis], lag_ratio=0.08), run_time=1.6)
        self.play(Write(terra_lab))

        # faixa vertical 0–100 km com gradiente de temperatura
        sx_c, sw, sh = 2.3, 2.0, 5.2
        y0s, y1s = -2.6, 2.6
        left, right = sx_c - sw / 2, sx_c + sw / 2
        n = 120
        faixa = VGroup(*[
            Rectangle(width=sw, height=sh / n * 1.03, stroke_width=0, fill_opacity=1,
                      fill_color=cor_T(temp((i + 0.5) / n * 100000)))
            .move_to([sx_c, y0s + (i + 0.5) * sh / n, 0])
            for i in range(n)
        ])
        borda = Rectangle(width=sw, height=sh, stroke_color=MUT, stroke_width=1.5,
                          fill_opacity=0).move_to([sx_c, (y0s + y1s) / 2, 0])

        ang = PI / 3
        mini = faixa.copy().scale(0.055).move_to(ec + (R_E + 0.13) * np.array([np.cos(ang), np.sin(ang), 0]))
        p_top = ec + (R_E + 0.27) * np.array([np.cos(ang + 0.07), np.sin(ang + 0.07), 0])
        p_bot = ec + (R_E + 0.01) * np.array([np.cos(ang - 0.07), np.sin(ang - 0.07), 0])
        con = VGroup(
            Line(p_top, [left, y1s, 0], stroke_width=1.2, color=MUT, stroke_opacity=0.45),
            Line(p_bot, [left, y0s, 0], stroke_width=1.2, color=MUT, stroke_opacity=0.45),
        )

        self.play(FadeIn(mini))
        self.play(ReplacementTransform(mini, faixa), Create(con), run_time=1.8)
        self.play(Create(borda))

        # linhas pontilhadas + rótulos com anti-colisão e linhas-guia
        ys = [y0s + sh * (h / 100000) for h in HL[1:]]
        ly = list(ys); gap = 0.55
        for i in range(1, len(ly)):
            ly[i] = max(ly[i], ly[i - 1] + gap)
        if ly[-1] > y1s - 0.05:
            ly[-1] = y1s - 0.05
            for i in range(len(ly) - 2, -1, -1):
                ly[i] = min(ly[i], ly[i + 1] - gap)

        tracejadas, rotulos, guias = VGroup(), VGroup(), VGroup()
        lx = right + 0.38
        for i, h in enumerate(HL[1:]):
            y = ys[i]
            tracejadas.add(DashedLine([left, y, 0], [right, y, 0], dash_length=0.07,
                                      stroke_width=2, color=WHITE, stroke_opacity=0.85))
            T = temp(h)
            l1 = Text(f"{int(h/1000)} km · {NOMES[i+1]}", font_size=15, color=INK)
            l2 = Text(f"{T:.1f} K", font_size=14, color=cor_T(T))
            par = VGroup(l1, l2).arrange(DOWN, buff=0.05, aligned_edge=LEFT)
            par.move_to([lx + par.width / 2, ly[i], 0])
            rotulos.add(par)
            if abs(ly[i] - y) > 0.02:
                guias.add(VGroup(
                    Line([right + 0.03, y, 0], [right + 0.20, y, 0],
                         stroke_width=1.2, color=PURPLE, stroke_opacity=0.6),
                    Line([right + 0.20, y, 0], [lx - 0.08, ly[i], 0],
                         stroke_width=1.2, color=PURPLE, stroke_opacity=0.6),
                ))

        ext = VGroup(
            Text("0 km", font_size=14, color=MUT).move_to([left - 0.42, y0s, 0]),
            Text("100 km", font_size=14, color=MUT).move_to([left - 0.52, y1s, 0]),
        )

        self.play(LaggedStart(*[Create(d) for d in tracejadas], lag_ratio=0.12), run_time=1.4)
        self.play(Write(ext))
        self.play(LaggedStart(*[Write(r) for r in rotulos], lag_ratio=0.12), run_time=2.2)
        if len(guias):
            self.play(Create(guias), run_time=0.8)

        nb, lw = 40, 3.0
        barra = VGroup(*[
            Rectangle(width=lw / nb * 1.05, height=0.16, stroke_width=0, fill_opacity=1,
                      fill_color=cor_T(180 + (290 - 180) * (i + 0.5) / nb))
            .move_to([-6.3 + lw / nb * (i + 0.5), -2.95, 0])
            for i in range(nb)
        ])
        leg = VGroup(
            Text("180 K · frio", font_size=13, color="#22d3ee").move_to([-6.3 + 0.55, -3.22, 0]),
            Text("290 K · quente", font_size=13, color="#ef4444").move_to([-3.3 - 0.70, -3.22, 0]),
        )
        self.play(FadeIn(barra), Write(leg))

        # c = cap("Zoom na casca de 0–100 km: a atmosfera é finíssima frente ao raio da Terra.\n"
        #         "Cor = temperatura real da camada — azul frio, vermelho quente.")
        self.play(Write(c)); self.wait(3.0)

Manim Community v0.22.1

In [29]:
%%manim -qm -v WARNING CenaDensidadeEDO
class CenaDensidadeEDO(Scene):
    def construct(self):
        title(self, "3 · Densidade — a EDO")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"\frac{dP}{dh}=-\rho(h)\,g(h)",
            "Equilíbrio hidrostático: subir dh reduz a pressão pelo peso da camada de ar acima.")
        c,_,_ = step(self,b,c, r"P=\rho\,\frac{R_g}{M_m}\,T",
            "Gás ideal: P, ρ e T se ligam pela constante específica do ar R_s = R_g/M_m.")
        c,_,_ = step(self,b,c,
            r"\frac{dP}{dh}=\frac{R_g}{M_m}\left(\frac{d\rho}{dh}\,T+\rho\,\frac{dT}{dh}\right)",
            "Derivando o gás ideal em h (regra do produto), com T variando.")
        c,_,_ = step(self,b,c,
            r"\frac{R_g}{M_m}\left(\rho'\,T+\rho\,\beta\right)=-\rho\,g",
            "Igualando as duas expressões de dP/dh e usando dT/dh = β.")
        c,eq,bx = step(self,b,c,
            r"\frac{d\rho}{dh}=-\rho\left(\frac{g\,M_m}{R_g\,T}+\frac{\beta}{T}\right)",
            "Isolando ρ': a EDO do notebook. 1º termo = peso; 2º = expansão térmica.",
            color=CYAN, box=True, font=36)
        c,_,_ = step(self,b,c,
            r"\frac{d\rho}{\rho}=-f(h)\,dh\;\Rightarrow\;\ln\rho=-\int f(h)\,dh",
            "Separação de variáveis: a solução é a exponencial de uma integral.")
        self.wait(1.5)

Manim Community v0.22.1

In [30]:
%%manim -qm -v WARNING CenaDensidadeSolucao
import numpy as np

# ---------------- paleta e helpers locais (autossuficiente) ----------------
INK = "#e9eef8"; MUT = "#93a1ba"; CYAN = "#22d3ee"; PURPLE = "#a06bff"
AMBER = "#ffb02e"; GREEN = "#3ddc84"; RED = "#ff4d6d"; PINK = "#e879f9"

ISO = ["\\rho", "\\beta", "K_i", "K_0", "g_0", "M_m", "R_g", "h_i", "Q", "A", "I"]

def _mt(tex, fs=36, color=INK):
    return MathTex(tex, font_size=fs, color=color, substrings_to_isolate=ISO)

def _fit(m, w=13.0):
    if m.width > w:
        m.scale(w / m.width)
    return m

def _cap(text, color=MUT):
    t = Text(text, font_size=21, color=color, line_spacing=0.8)
    _fit(t, 13.2)
    return t.to_edge(DOWN, buff=0.32)

def _title(scene, txt):
    t = Text(txt, font_size=30, color=CYAN, weight="BOLD").to_edge(UP, buff=0.26)
    scene.play(Write(t), run_time=0.8)
    return t

class Work:
    """Área de rascunho abaixo da equação principal."""
    def __init__(self, top=0.95, left=-6.4, w=12.8, gap=0.24):
        self.y = top; self.left = left; self.w = w; self.gap = gap
        self.g = VGroup()
    def line(self, tex, fs=32, color=INK):
        m = _fit(_mt(tex, fs, color), self.w)
        m.move_to([self.left + m.width / 2, self.y - m.height / 2, 0])
        self.y -= m.height + self.gap
        self.g.add(m)
        return m

def _clear(scene, wk):
    if len(wk.g):
        scene.play(FadeOut(wk.g), run_time=0.5)
    return Work()

def _say(scene, old, txt, color=MUT):
    new = _cap(txt, color)
    if old is None:
        scene.play(FadeIn(new), run_time=0.4)
    else:
        scene.play(FadeOut(old), FadeIn(new), run_time=0.45)
    return new

def _main(scene, old, tex, fs=36, color=INK, run=1.4):
    """Morfa a equação principal. ReplacementTransform alinha as famílias
    sozinho — não usa o ghost_to do TransformMatchingTex (bug do zip strict)."""
    new = _fit(_mt(tex, fs, color), 13.2)
    new.move_to(UP * 2.1 if old is None else old.get_center())
    if old is None:
        scene.play(Write(new), run_time=run)
    else:
        try:
            scene.play(ReplacementTransform(old, new), run_time=run)
        except Exception:
            scene.play(FadeOut(old), FadeIn(new), run_time=run)
    return new

def _expand(scene, symbol, tex, at, fs=34, color=AMBER):
    """O símbolo 'abre' e vira a expressão expandida (efeito de substituição)."""
    tgt = _fit(_mt(tex, fs, color), 12.4).move_to(at)
    try:
        scene.play(ReplacementTransform(symbol.copy(), tgt), run_time=1.1)
    except Exception:
        scene.play(FadeIn(tgt), run_time=1.1)
    scene.play(Circumscribe(tgt, color=color, buff=0.12), run_time=0.8)
    return tgt

def _card(tex, label, color=CYAN, fs=30):
    m = _mt(tex, fs, color)
    lab = Text(label, font_size=15, color=MUT)
    body = VGroup(lab, m).arrange(DOWN, buff=0.14, aligned_edge=LEFT)
    box = RoundedRectangle(corner_radius=0.14, width=body.width + 0.5,
                           height=body.height + 0.4, stroke_color=color,
                           stroke_width=1.5, fill_color="#0a1120", fill_opacity=0.9)
    box.move_to(body)
    return VGroup(box, body)

# ---------------- F e K exatamente como no notebook ----------------
Ac = R**2 * G0 * MM / RG

def Fcode(h, i):
    b = BETA[i]; hi = HL[i]; T = temp(h)
    if b != 0:
        C = temp(hi)
        return (Ac * (b * (R + h) * np.log((-b * hi + b * h + C) / (R + h))
                      + R * b + b * hi - C)
                / ((R + h) * (b * R + b * hi - C)**2) + np.log(T))
    return -MM * G0 * R**2 / (RG * T * (R + h))

Kc = [-np.log(RHO0) - Fcode(0.0, 0)]
for i in range(1, 8):
    Kc.append(Kc[i - 1] + Fcode(HL[i], i - 1) - Fcode(HL[i], i))


class CenaDensidadeSolucao(Scene):
    def construct(self):
        _title(self, "4 · Densidade — da EDO à solução analítica")
        wk = Work(); c = None

        # ================= ATO 1 — ponto de partida =================
        eq = _main(self, None,
            r"\frac{d\rho}{dh}=-\rho\left(\frac{g\,M_m}{R_g\,T}+\frac{1}{T}\frac{dT}{dh}\right)")
        c = _say(self, c, "Equilíbrio hidrostático + gás ideal. Dentro da camada i: g depende de h, T é linear em h.")
        self.wait(0.8)

        gcard = _card(r"g(h)=g_0\frac{R^2}{(R+h)^2}", "gravidade", PURPLE).move_to([-3.4, 0.2, 0])
        bcard = _card(r"\frac{dT}{dh}=\beta", "gradiente da camada", AMBER).move_to([3.0, 0.2, 0])
        self.play(FadeIn(gcard), FadeIn(bcard))
        self.wait(0.6)

        eq = _main(self, eq,
            r"\frac{d\rho}{dh}=-\rho\left(\frac{g_0R^2}{(R+h)^2}\cdot\frac{M_m}{R_g\,T}+\frac{\beta}{T}\right)", fs=34)
        eq.set_color_by_tex("g_0", PURPLE); eq.set_color_by_tex("\\beta", AMBER)
        self.play(Indicate(eq, color=CYAN, scale_factor=1.04))
        c = _say(self, c, "Abrimos g(h) e trocamos dT/dh por β. Tudo agora está em função de h.")
        self.play(FadeOut(gcard), FadeOut(bcard))

        # ================= ATO 2 — constante A e separação =================
        Adef = _expand(self, eq, r"A\equiv\frac{R^2g_0M_m}{R_g}", [0, 0.35, 0], color=CYAN)
        c = _say(self, c, "Agrupamos as constantes num único símbolo A — é o 'A' do seu código.")
        eq = _main(self, eq,
            r"\frac{d\rho}{dh}=-\rho\left(\frac{A}{(R+h)^2\,T}+\frac{\beta}{T}\right)")
        eq.set_color_by_tex("A", CYAN)
        self.play(FadeOut(Adef))

        eq = _main(self, eq,
            r"\frac{d\rho}{\rho}=-\left(\frac{A}{(R+h)^2\,T}+\frac{\beta}{T}\right)dh")
        c = _say(self, c, "Separação de variáveis: ρ de um lado, h do outro. Integrar é imediato.")
        self.wait(0.6)

        # ================= ATO 3 — integração e o termo ln T =================
        eq = _main(self, eq,
            r"\ln\rho=-\left(A\int\frac{dh}{(R+h)^2\,T}+\ln T+K_i\right)", fs=34)
        w1 = wk.line(r"\int\frac{\beta}{T}\,dh=\int\frac{T'(h)}{T(h)}\,dh=\ln T", 30, AMBER)
        self.play(Write(w1))
        c = _say(self, c, "O segundo termo é a derivada do logaritmo: ∫T′/T = ln T. K_i é a constante da camada.")
        self.play(Circumscribe(w1, color=AMBER, buff=0.1))
        self.wait(0.7)

        Idef = _expand(self, eq, r"I=\int\frac{dh}{(R+h)^2\,T}", [0, -0.55, 0], color=GREEN)
        eq = _main(self, eq, r"\ln\rho=-\big(A\,I+\ln T+K_i\big)")
        eq.set_color_by_tex("I", GREEN)
        c = _say(self, c, "Toda a dificuldade está em I. Vamos resolvê-la nos dois casos: β = 0 e β ≠ 0.")
        self.play(FadeOut(Idef))
        wk = _clear(self, wk)

        # ================= ATO 4 — caso β = 0 =================
        eq = _main(self, eq, r"\beta=0\;\Rightarrow\;T=C\;\;\Rightarrow\;\;I=\frac{1}{C}\int\frac{dh}{(R+h)^2}", fs=33)
        c = _say(self, c, "Camada isotérmica: T sai da integral e ∫dh/(R+h)² = −1/(R+h).", AMBER)
        w1 = wk.line(r"I=-\frac{1}{C\,(R+h)}", 32)
        self.play(Write(w1))
        w2 = wk.line(r"A\,I=-\frac{M_mg_0R^2}{R_g\,T\,(R+h)}", 32, CYAN)
        self.play(Write(w2))
        w3 = wk.line(r"\ln T=\text{const}\;\Rightarrow\;\text{absorvido em }K_i", 28, MUT)
        self.play(Write(w3))
        c = _say(self, c, "Por isso o ramo β = 0 do seu F não tem o + ln T: ele é constante e some dentro de K_i.", AMBER)
        self.wait(1.0)
        box0 = SurroundingRectangle(w2, color=CYAN, buff=0.14)
        self.play(Create(box0))
        self.wait(0.6)
        self.play(FadeOut(box0))
        wk = _clear(self, wk)

        # ================= ATO 5 — caso β ≠ 0: u e Q =================
        eq = _main(self, eq, r"T=C+\beta\,(h-h_i)", fs=36)
        c = _say(self, c, "Com β ≠ 0, T é linear. O truque é escrever T em função de u = R + h.")
        eq = _main(self, eq, r"T=\beta(R+h)-\big(\beta R+\beta h_i-C\big)", fs=34)
        c = _say(self, c, "Somando e subtraindo βR: a mesma reta, reescrita para expor (R+h).")
        self.wait(0.6)

        udef = _expand(self, eq, r"u=R+h", [-3.6, 0.3, 0], color=PURPLE)
        Qdef = _expand(self, eq, r"Q=\beta R+\beta h_i-C", [3.2, 0.3, 0], color=PINK)
        c = _say(self, c, "Dois símbolos novos: u (deslocamento do centro da Terra) e Q (constante da camada).")
        eq = _main(self, eq, r"T=\beta u-Q", fs=38)
        eq.set_color_by_tex("Q", PINK)
        self.play(FadeOut(udef), FadeOut(Qdef))

        eq = _main(self, eq, r"I=\int\frac{du}{u^2\,(\beta u-Q)}", fs=36)
        c = _say(self, c, "A integral virou racional em u: denominador u²(βu − Q). Frações parciais resolvem.")
        self.wait(0.7)

        # ================= ATO 6 — frações parciais =================
        eq = _main(self, eq,
            r"\frac{1}{u^2(\beta u-Q)}=\frac{a}{u}+\frac{b}{u^2}+\frac{e}{\beta u-Q}", fs=34)
        w1 = wk.line(r"1=a\,u(\beta u-Q)+b(\beta u-Q)+e\,u^2", 30)
        self.play(Write(w1))
        c = _say(self, c, "Multiplicando pelo denominador comum, obtemos uma identidade válida para todo u.")

        e1 = wk.line(r"u=0:\;\;1=-bQ\;\Rightarrow\;b=-\frac{1}{Q}", 29, AMBER)
        e2 = wk.line(r"u=\frac{Q}{\beta}:\;\;1=e\frac{Q^2}{\beta^2}\;\Rightarrow\;e=\frac{\beta^2}{Q^2}", 29, AMBER)
        e3 = wk.line(r"[u^2]:\;\;0=a\beta+e\;\Rightarrow\;a=-\frac{\beta}{Q^2}", 29, AMBER)
        for e in (e1, e2, e3):
            self.play(Write(e), run_time=0.9)
            self.play(Circumscribe(e, color=AMBER, buff=0.08), run_time=0.6)
        c = _say(self, c, "Avaliar em u = 0 e u = Q/β mata dois termos de cada vez; o coeficiente de u² fecha o terceiro.")
        self.wait(0.8)

        eq = _main(self, eq,
            r"\frac{1}{u^2(\beta u-Q)}=-\frac{\beta}{Q^2}\frac{1}{u}-\frac{1}{Q}\frac{1}{u^2}+\frac{\beta^2}{Q^2}\frac{1}{\beta u-Q}", fs=30)
        wk = _clear(self, wk)
        c = _say(self, c, "Cada parcela agora integra em logaritmo ou em 1/u.")

        # ================= ATO 7 — integrar e recombinar =================
        eq = _main(self, eq,
            r"I=-\frac{\beta}{Q^2}\ln u+\frac{1}{Qu}+\frac{\beta}{Q^2}\ln(\beta u-Q)", fs=33)
        c = _say(self, c, "∫du/u = ln u ; ∫du/u² = −1/u ; ∫du/(βu−Q) = (1/β)ln(βu−Q).")
        self.wait(0.7)
        eq = _main(self, eq, r"I=\frac{\beta}{Q^2}\ln\frac{T}{u}+\frac{1}{Qu}", fs=36)
        c = _say(self, c, "Os dois logaritmos se combinam e usamos βu − Q = T de volta.")
        eq = _main(self, eq, r"I=\frac{\beta u\,\ln\frac{T}{u}+Q}{u\,Q^2}", fs=36)
        c = _say(self, c, "Denominador comum uQ²: esta é a forma que aparece no seu código.")
        self.wait(0.7)

        eq = _main(self, eq, r"F=A\cdot\frac{\beta u\,\ln\frac{T}{u}+Q}{u\,Q^2}+\ln T", fs=35)
        eq.set_color_by_tex("A", CYAN)
        c = _say(self, c, "F ≡ A·I + ln T. Falta só devolver u, Q e T às variáveis originais.")
        self.wait(0.8)

        # ================= ATO 8 — abrindo u, Q, T e A no F final =================
        legend = VGroup(
            _card(r"u\;\to\;R+h", "volta 1", PURPLE, 26),
            _card(r"Q\;\to\;\beta R+\beta h_i-C", "volta 2", PINK, 26),
            _card(r"T\;\to\;-\beta h_i+\beta h+C", "volta 3", AMBER, 26),
        ).arrange(RIGHT, buff=0.35).move_to([0, 0.15, 0])
        self.play(LaggedStart(*[FadeIn(x) for x in legend], lag_ratio=0.2))
        self.wait(0.8)

        eq = _main(self, eq,
            r"F(h,i)=\frac{R^2g_0M_m}{R_g}\cdot"
            r"\frac{\beta(R+h)\ln\left(\frac{-\beta h_i+\beta h+C}{R+h}\right)+R\beta+\beta h_i-C}"
            r"{(R+h)\left(\beta R+\beta h_i-C\right)^2}+\ln T", fs=30, run=2.0)
        self.play(FadeOut(legend))
        boxF = SurroundingRectangle(eq, color=CYAN, buff=0.16)
        self.play(Create(boxF))
        c = _say(self, c, "Idêntico ao F(h, i) do notebook — inclusive o sinal de −βh_i + βh + C dentro do log.", CYAN)
        self.wait(1.2)

        code = VGroup(
            Text("b  = beta[i]", font_size=17, color=PINK),
            Text("hi = alturas[i]", font_size=17, color=PURPLE),
            Text("C  = temperatura(hi)", font_size=17, color=AMBER),
            Text("Q  = b*R + b*hi - C", font_size=17, color=CYAN),
        ).arrange(DOWN, buff=0.12, aligned_edge=LEFT).move_to([-4.6, -0.5, 0])
        cb = RoundedRectangle(corner_radius=0.12, width=code.width + 0.5,
                              height=code.height + 0.4, stroke_color=MUT,
                              stroke_width=1.2, fill_color="#0a1120", fill_opacity=0.9).move_to(code)
        self.play(FadeIn(cb), Write(code))
        c = _say(self, c, "Mapa direto para o código: Q é exatamente (b*R + b*hi − C), o termo que aparece ao quadrado.")
        self.wait(1.2)
        self.play(FadeOut(cb), FadeOut(code), FadeOut(boxF))

        # ================= ATO 9 — de ln ρ para ρ =================
        eq = _main(self, eq, r"\ln\rho(h)=-\big(F(h,i)+K_i\big)", fs=38)
        eq = _main(self, eq, r"\rho(h)=\exp\!\big[-\big(F(h,i)+K_i\big)\big]", fs=38, color=GREEN)
        c = _say(self, c, "Exponenciando: é o return np.exp(-(F(h,i) + k)) do seu rho().", GREEN)
        self.wait(0.8)

        # ================= ATO 10 — K0 e a cascata =================
        eq = _main(self, eq, r"\rho(h_0)=\rho_0\;\Rightarrow\;\ln\rho_0=-\big(F(h_0,0)+K_0\big)", fs=33)
        eq = _main(self, eq, r"K_0=-\ln\rho_0-F(h_0,0)", fs=38, color=AMBER)
        k0v = _mt(rf"K_0={Kc[0]:.4f}", 32, AMBER).move_to([0, 0.3, 0])
        self.play(Write(k0v))
        c = _say(self, c, "A primeira constante ancora a solução no nível do mar: ρ₀ = 1,225 kg/m³.")
        self.wait(0.8)
        self.play(FadeOut(k0v))

        eq = _main(self, eq, r"F(h_i,i-1)+K_{i-1}=F(h_i,i)+K_i", fs=36)
        c = _say(self, c, "Na fronteira h_i a densidade não pode saltar: o valor vindo de baixo = o valor de cima.")
        eq = _main(self, eq, r"K_i=K_{i-1}+F(h_i,i-1)-F(h_i,i)", fs=38, color=PURPLE)
        c = _say(self, c, "Isolando K_i: cada camada herda a anterior. É o seu laço for i in range(1, len(beta)).", PURPLE)
        self.wait(1.0)

        kt = VGroup(*[
            _mt(rf"K_{i}={Kc[i]:.4f}", 26, INK if i else AMBER)
            for i in range(8)
        ]).arrange_in_grid(rows=4, cols=2, buff=(0.22, 1.1)).move_to([0, -0.5, 0])
        self.play(LaggedStart(*[FadeIn(x) for x in kt], lag_ratio=0.08))
        c = _say(self, c, "Os oito K do notebook, reproduzidos pela fórmula. K₀ = −52,4875 confere.")
        self.wait(1.4)
        self.play(FadeOut(kt), FadeOut(eq), FadeOut(c))

                # ================= ATO 11 — verificação gráfica =================
        # ρ = exp(...) > 0 sempre. O eixo vertical é log10(ρ): negativo é o
        # expoente (ordem de grandeza), nunca a densidade.
        pos = _mt(r"\rho(h)=e^{-\left(F(h,i)+K_i\right)}>0", 36, GREEN).move_to(UP * 2.1)
        self.play(Write(pos))
        c = _cap("ρ é uma exponencial → nunca negativa. O que fica negativo é log₁₀ρ, não ρ.", AMBER)
        self.play(Write(c)); self.wait(1.8)
        self.play(FadeOut(pos), FadeOut(c))

        ax = Axes(x_range=[0, 100, 100], y_range=[-8, 1, 2],
                  x_length=10.4, y_length=3.6, tips=False,
                  axis_config={"color": MUT, "stroke_width": 2}).shift(DOWN * 0.35)
        # a linha horizontal em log ρ = 0 é ρ = 1 kg/m³, NÃO o zero → escondemos
        ax.x_axis.set_opacity(0)

        P = lambda km, e: ax.coords_to_point(km, e)
        yb, yt = P(0, -8)[1], P(0, 1)[1]
        xl, xr = P(0, -8)[0], P(100, -8)[0]

        base = Line([xl, yb, 0], [xr, yb, 0], stroke_width=2, color=MUT)

        # ticks verticais como potências de 10 (nada de números "negativos")
        grade = VGroup()
        for e in (0, -2, -4, -6, -8):
            yy = P(0, e)[1]
            grade.add(DashedLine([xl, yy, 0], [xr, yy, 0], dash_length=0.06,
                                 stroke_width=1, color=MUT, stroke_opacity=0.22))
            lab_e = MathTex(f"10^{{{e}}}", font_size=23, color=MUT)
            lab_e.next_to(np.array([xl, yy, 0]), LEFT, buff=0.16)
            grade.add(lab_e)

        xnums = VGroup()
        for km in (0, 20, 40, 60, 80, 100):
            t = Text(str(km), font_size=17, color=MUT)
            t.next_to(np.array([P(km, -8)[0], yb, 0]), DOWN, buff=0.14)
            xnums.add(t)

        xlab = MathTex(r"h\;(\mathrm{km})", font_size=25, color=MUT)
        xlab.next_to(np.array([xr, yb, 0]), RIGHT, buff=0.25)
        ylab = MathTex(r"\rho\;(\mathrm{kg/m^3})", font_size=25, color=MUT)
        ylab.move_to([xl + ylab.width / 2 - 0.4, yt + 0.38, 0])
        note = Text("escala log₁₀", font_size=16, color=AMBER).next_to(ylab, RIGHT, buff=0.28)

        vlines = VGroup(*[
            DashedLine([P(h / 1000, -8)[0], yb, 0], [P(h / 1000, -8)[0], yt, 0],
                       dash_length=0.08, stroke_width=1.6,
                       color=PURPLE, stroke_opacity=0.7)
            for h in HL[1:]
        ])

        xs = [0.0] + [h / 1000 for h in HL[1:]] + [100.0]
        pts = []
        for a, b in zip(xs[:-1], xs[1:]):
            for t in np.linspace(0, 1, 26, endpoint=False):
                km = a + (b - a) * t
                pts.append(P(km, np.log10(rho(km * 1000))))
        pts.append(P(100, np.log10(rho(100000))))
        cur = VMobject(stroke_color=GREEN, stroke_width=4)
        cur.set_points_as_corners(pts)

        self.play(Create(ax), Create(base), Write(grade), run_time=1.2)
        self.play(Write(xnums), Write(xlab), Write(ylab), Write(note))
        self.play(LaggedStart(*[Create(v) for v in vlines], lag_ratio=0.1), run_time=1.2)
        self.play(Create(cur), run_time=2.2)

        # valores REAIS de ρ (todos positivos) sobre a curva
        def _sci(v):
            if v >= 0.01:
                return f"{v:.3f}"
            e = int(np.floor(np.log10(v)))
            return rf"{v / 10**e:.2f}\times10^{{{e}}}"

        marks = VGroup()
        for km, col in ((0, CYAN), (11, GREEN), (50, AMBER), (86, PINK)):
            v = float(rho(km * 1000))
            d = Dot(P(km, np.log10(v)), color=col, radius=0.065)
            t = MathTex(_sci(v), font_size=22, color=col)
            t.next_to(d, UP, buff=0.16)
            if km >= 80:
                t.shift(LEFT * 0.7)
            marks.add(VGroup(d, t))
        self.play(LaggedStart(*[FadeIn(m) for m in marks], lag_ratio=0.18), run_time=1.6)

        c = _cap("Nenhum valor negativo: −6 no eixo significa 10⁻⁶ kg/m³. "
                 "Cada trecho é uma camada com seu próprio F e K.", GREEN)
        self.play(Write(c))
        self.wait(3.0)

Manim Community v0.22.1

In [23]:
%%manim -qm -v WARNING CenaAero
class CenaAero(Scene):
    def construct(self):
        title(self, "5 · Som, Mach e arrasto")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"V_s^2=\left(\frac{\partial P}{\partial\rho}\right)_s",
            "O som é uma perturbação tão rápida que não troca calor: deriva adiabática.")
        c,_,_ = step(self,b,c, r"P\propto\rho^\gamma\;\Rightarrow\;\frac{\partial P}{\partial\rho}=\gamma\frac{P}{\rho}",
            "Num processo adiabático, P/ρ^γ é constante.")
        c,eq,bx = step(self,b,c, r"V_s=\sqrt{\gamma\,\frac{R_g}{M_m}\,T(h)}",
            "Com P/ρ = R_g T/M_m, o som só depende de T: 295,07 m/s a 11 km.", color=CYAN, box=True)
        c,_,_ = step(self,b,c, r"M=\frac{|v|}{V_s}",
            "Mach compara a velocidade do foguete com o som local.")
        c,_,_ = step(self,b,c, r"q=\tfrac12\,\rho\,|v|^2",
            "Pressão dinâmica: energia cinética por volume do escoamento.")
        c,eq,bx = step(self,b,c, r"\vec F_D=-\tfrac12\,C_D(M)\,\rho(h)\,A\,|v|\,\vec v",
            "Arrasto = q·A·C_D contra a velocidade; C_D da tabela transônica (pico ≈0,55 em M≈1,5).",
            color=CYAN, box=True, font=36)
        self.wait(1.5)

Manim Community v0.22.1

In [15]:
%%manim -qm -v WARNING CenaEmpuxo
class CenaEmpuxo(Scene):
    def construct(self):
        title(self, "6 · Empuxo do motor")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"T=\dot m\,\mu_e+(P_c-P_a)A_b",
            "Balanço de momentum no bocal: fluxo de momentum + termo de pressão na saída.")
        c,_,_ = step(self,b,c,
            r"\mu_e=\sqrt{2c_pT_c\left(1-\left(\frac{P_a}{P_c}\right)^{\frac{\gamma-1}{\gamma}}\right)}",
            "Expansão isentrópica: entalpia vira velocidade (h0 = h + v²/2).", font=34)
        c,_,_ = step(self,b,c, r"c_p=\frac{\gamma R_s}{\gamma-1}",
            "Com cp do gás (γ=1,15; M=23 g/mol, LOX/RP-1) chega-se ao μe do notebook.")
        c,_,_ = step(self,b,c,
            r"\dot m=\frac{P_cA_t}{\sqrt{T_c}}\sqrt{\frac{\gamma}{R_s}}\left(\frac{2}{\gamma+1}\right)^{\frac{\gamma+1}{2(\gamma-1)}}",
            "Escoamento estrangulado (Mach 1 na garganta): a vazão fixa a pressão da câmara.", font=30)
        c,eq,bx = step(self,b,c,
            r"P_c=\frac{\dot m}{A_t}\sqrt{\frac{R_sT_c}{\gamma}}\left(\frac{\gamma+1}{2}\right)^{\frac{\gamma+1}{2(\gamma-1)}}",
            "Invertendo a relação anterior — exatamente p_camara_combustao.", color=CYAN, box=True, font=32)
        b=clear(self,b)
        c,_,_ = step(self,b,c, r"T_c=T_w+(T_{ad}-T_w)\frac{\dot m}{\dot m+0.05\,\dot m}",
            "Modelo de câmara do notebook: resfriamento ∝ vazão → T_c ≈ 3371 K.")
        c,eq,bx = step(self,b,c, r"T=c\left(\dot m\,\mu_e+(P_c-P_a)A_b\right)",
            "Forma final com o coeficiente de correção c do bocal. Em(0) ≈ 37,78 MN.",
            color=CYAN, box=True)
        self.wait(1.5)

Manim Community v0.22.1

In [16]:
%%manim -qm -v WARNING CenaSeparacao
class CenaSeparacao(Scene):
    def construct(self):
        title(self, "7 · Separação de estágios")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"M_1v_1=M_2v_2",
            "Momentum conservado no desacoplamento: momenta iguais e opostos (referencial do CM).")
        c,_,_ = step(self,b,c, r"E=\tfrac12M_1v_1^2+\tfrac12M_2v_2^2",
            "A carga do desacoplador injeta energia cinética E no sistema.")
        c,_,_ = step(self,b,c, r"v_2=\frac{M_1}{M_2}\,v_1",
            "Isolando v2 do momentum para substituir na energia.")
        c,_,_ = step(self,b,c, r"E=\tfrac12\,v_1^2\,\frac{M_1(M_1+M_2)}{M_2}",
            "Substituindo v2 na energia e fatorando.")
        c,_,_ = step(self,b,c, r"\Delta V=v_1=\sqrt{\frac{2E\,M_2}{M_1(M_1+M_2)}}",
            "Isolando v1: o ganho de velocidade do módulo principal.")
        c,eq,bx = step(self,b,c, r"\Delta V=\sqrt{\frac{M_2\,E}{M_1(M_1+M_2)}}",
            "No notebook o fator 2 é absorvido na definição de E (E=10 kJ): forma de v_momento.",
            color=CYAN, box=True)
        self.wait(1.5)

Manim Community v0.22.1

In [17]:
%%manim -qm -v WARNING CenaIntegracao
class CenaIntegracao(Scene):
    def construct(self):
        title(self, "8 · Integração trapezoidal (Heun)")
        b=Board(); c=None
        c,_,_ = step(self,b,c, r"a_n=\frac{F(h_n,v_n,m_n)}{m_n}",
            "A resultante (peso + empuxo + arrasto + Arquimedes) dá a aceleração a cada passo.")
        c,_,_ = step(self,b,c, r"\tilde v=v_n+a_n\,dt",
            "Como a_{n+1} depende de v_{n+1}, fazemos uma previsão de Euler (preditor).")
        c,_,_ = step(self,b,c, r"v_{n+1}=v_n+\frac{dt}{2}\left(a_n+a_{n+1}\right)",
            "Corretor: média das acelerações nas duas pontas — regra do trapézio.")
        c,_,_ = step(self,b,c, r"h_{n+1}=h_n+\frac{dt}{2}\left(v_n+v_{n+1}\right)",
            "A posição integra a velocidade do mesmo jeito: erro O(dt³) por passo, O(dt²) global.")
        c,_,_ = step(self,b,c, r"m_{n+1}=m_n-\frac{dt}{2}\left(\dot m_n+\dot m_{n+1}\right)",
            "A massa cai pela vazão média; dt = 1/60 s no notebook.")
        c,eq,bx = step(self,b,c, r"cg\ \geq\ \sum mc\ \Rightarrow\ \text{ejeção}",
            "Quando o consumo acumulado atinge o combustível do estágio, aplica-se Δv e troca-se de estágio.",
            color=CYAN, box=True)
        self.wait(1.5)

Manim Community v0.22.1

In [21]:
%%manim -qm -v WARNING CenaDiferencasFinitas
import math
import numpy as np
from pathlib import Path
from PIL import Image

# ================= paleta + helpers locais =================
INK="#e9eef8"; MUT="#93a1ba"; CYAN="#22d3ee"; PURPLE="#a06bff"
AMBER="#ffb02e"; GREEN="#3ddc84"; RED="#ff4d6d"; PINK="#e879f9"; ORANGE="#ff7a1c"

def _fit(m, w=12.8):
    if m.width > w: m.scale(w / m.width)
    return m

def _mt(tex, fs=34, color=INK):
    return _fit(MathTex(tex, font_size=fs, color=color))

def _cap(txt, color=MUT):
    return _fit(Text(txt, font_size=20, color=color), 12.6).to_edge(DOWN, buff=0.30)

def _title(scene, txt):
    t = Text(txt, font_size=29, color=CYAN, weight="BOLD").to_edge(UP, buff=0.24)
    scene.play(Write(t), run_time=0.7); return t

def _say(scene, old, txt, color=MUT):
    new = _cap(txt, color)
    if old is None: scene.play(FadeIn(new), run_time=0.35)
    else: scene.play(FadeOut(old), FadeIn(new), run_time=0.4)
    return new

def _morph(scene, old, tex, fs=34, color=INK, at=None, run=1.3):
    new = _mt(tex, fs, color)
    new.move_to(at if at is not None else (UP*2.45 if old is None else old.get_center()))
    if old is None: scene.play(Write(new), run_time=run)
    else:
        try: scene.play(ReplacementTransform(old, new), run_time=run)
        except Exception: scene.play(FadeOut(old), FadeIn(new), run_time=run)
    return new

def _poly(pts, color, w=3, op=1.0):
    v = VMobject(stroke_color=color, stroke_width=w, stroke_opacity=op)
    v.set_points_as_corners([np.array(p, float) for p in pts])
    return v

def _box(tex, label, color=CYAN, fs=27):
    m = MathTex(tex, font_size=fs, color=color)
    l = Text(label, font_size=13, color=MUT)
    body = VGroup(l, m).arrange(DOWN, buff=0.12)          # VGroup (não Group)
    bx = RoundedRectangle(corner_radius=0.12, width=body.width+0.44, height=body.height+0.34,
                          stroke_color=color, stroke_width=1.4,
                          fill_color="#0a1120", fill_opacity=0.92).move_to(body)
    return VGroup(bx, body)                              # VGroup (não Group)

# ================= sprite (compacta os vãos entre estágios) =================
def _rocket_icon(height=1.0):
    p = Path("media/assets/foguete.png")
    if p.exists():
        try:
            a = np.array(Image.open(p).convert("RGBA"))
            al = a[..., 3]; rgb = a[..., :3].astype(int)
            cn = np.array([a[0,0], a[0,-1], a[-1,0], a[-1,-1]])
            if cn[:, 3].min() < 200:
                mask = al > 40
            else:
                bg = np.median(cn[:, :3].astype(int), axis=0)
                mask = (al > 40) & (np.abs(rgb - bg).sum(axis=2) > 45)
            rows = mask.sum(axis=1) > max(2, int(a.shape[1]*0.002))
            idx = np.where(rows)[0]
            if len(idx) > 20:
                sub = a[idx]
                cols = np.where(mask[idx].any(axis=0))[0]
                sub = sub[:, cols[0]:cols[-1]+1]
                m = ImageMobject(sub); m.scale_to_fit_height(height); return m
        except Exception:
            pass
    g = VGroup()
    g.add(Polygon([0,0.52,0],[0.15,0.16,0],[-0.15,0.16,0], fill_color="#f1ede2",
                  fill_opacity=1, stroke_width=0))
    g.add(Rectangle(width=0.30, height=0.62, fill_color="#f1ede2", fill_opacity=1,
                    stroke_width=0).move_to([0,-0.15,0]))
    for sx in (-1,1):
        g.add(Polygon([sx*0.15,-0.30,0],[sx*0.28,-0.46,0],[sx*0.15,-0.46,0],
                      fill_color="#c9c3b4", fill_opacity=1, stroke_width=0))
    g.add(Polygon([-0.09,-0.46,0],[0.09,-0.46,0],[0.13,-0.58,0],[-0.13,-0.58,0],
                  fill_color="#3a4150", fill_opacity=1, stroke_width=0))
    g.scale_to_fit_height(height)
    return g

# ================= atmosfera escalar rápida (mesmos K do notebook) =================
_TB = [T0]
for i in range(7): _TB.append(_TB[i] + BETA[i]*(HL[i+1]-HL[i]))
_AC = R*R*G0*MM/RG

def _lay(h):
    i = 0
    while i < 7 and h >= HL[i+1]: i += 1
    return i

def _T(h):
    i = _lay(h); return _TB[i] + BETA[i]*(h - HL[i])

def _Fd(h, i):
    b = BETA[i]; hi = HL[i]; T = _T(h); C = _TB[i]
    if b != 0.0:
        num = b*(R+h)*math.log((b*(h-hi)+C)/(R+h)) + R*b + b*hi - C
        den = (R+h)*(b*R + b*hi - C)**2
        return _AC*num/den + math.log(T)
    return -MM*G0*R*R/(RG*T*(R+h))

_K = [-math.log(RHO0) - _Fd(0.0, 0)]
for i in range(1, 8): _K.append(_K[i-1] + _Fd(HL[i], i-1) - _Fd(HL[i], i))

def _rho(h):
    h = max(0.0, h); i = _lay(h)
    return math.exp(-(_Fd(h, i) + _K[i]))

def _g(h): return G0*(R/(R + max(0.0, h)))**2

# ================= motor + estágios (dados do notebook) =================
GG=1.15; MMG=0.023; RS=RG/MMG; TAD=3500.0; TW=800.0; COOL=0.05; ECC=10_000.0
STG=[dict(A=289.6015,V=12181.3369,ms=133583,mc=2148213,Ab=49.677),
     dict(A=79.4598, V=1973.8777, ms=38442, mc=437595, Ab=15.021),
     dict(A=34.3589, V=621.0478,  ms=17091, mc=101822, Ab=3.004)]
FLOW=[14000.0,1000.0,600.0]
MACH=[0,0.5,1,1.25,1.5,2,2.5,3,4,5,7,9,10]
CDT=[0.300,0.260,0.375,0.510,0.550,0.470,0.350,0.270,0.210,0.200,0.230,0.260,0.260]
DT=1/60

def _thrust(vm, Ab, r, T):
    tc = TW + (TAD-TW)/(1.0+COOL)
    pa = r*RG*T/MM
    pc = (vm/Ab)*math.sqrt(RS*tc/GG)*((GG+1)/2)**((GG+1)/(2*(GG-1)))
    rat = min(pa/max(pc,1.0), 1.0)
    mue = math.sqrt(max(0.0, (2*GG*RS*tc/(GG-1))*(1-rat**((GG-1)/GG))))
    return vm*mue + (pc-pa)*Ab

def _acc(h, v, m, st, vm, vol):
    sd = STG[min(st,2)]
    g = _g(h); r = _rho(h); T = _T(h)
    f = -m*g + r*g*vol
    if v != 0.0:
        vs = math.sqrt(1.4*RG*T/MM)
        Cd = float(np.interp(abs(v)/vs, MACH, CDT))
        f -= 0.5*Cd*r*sd["A"]*abs(v)*v
    if vm > 0.0: f += _thrust(vm, sd["Ab"], r, T)
    return f/m

def march(scheme="trap", dt=DT, tmax=800.0, every=12):
    st=0; fuel=[s["mc"] for s in STG]
    m=sum(s["ms"]+s["mc"] for s in STG); vol=sum(s["V"] for s in STG)
    h=0.0; v=0.0; t=0.0; k=0
    Ts=[0.0]; Hs=[0.0]; Vs=[0.0]; Ms=[m]; As=[]
    ev=[]
    As.append(_acc(h,v,m,st,FLOW[0],vol))
    while st < 3 and t < tmax:
        vm = FLOW[st] if fuel[st] > 0 else 0.0
        a1 = _acc(h,v,m,st,vm,vol)
        if scheme == "euler":
            vn = v + a1*dt; hn = h + v*dt
        else:
            vp = v + a1*dt; hp = h + v*dt
            a2 = _acc(hp,vp,m,st,vm,vol)
            vn = v + 0.5*dt*(a1+a2); hn = h + 0.5*dt*(v+vn)
        if hn < 0.0: hn=0.0; vn=0.0
        burn = min(vm*dt, fuel[st]); fuel[st] -= burn; m -= burn
        t += dt; k += 1
        if fuel[st] <= 0.0:
            if st < 2:
                M2=STG[st]["ms"]; M1=m-M2
                dv=math.sqrt(M2*ECC/(M1*(M1+M2)))
                vn += dv; m -= M2; vol -= STG[st]["V"]
                ev.append((t, st+1, dv)); st += 1
            else:
                ev.append((t, 3, 0.0)); st = 3
        h=hn; v=vn
        if k % every == 0:
            Ts.append(t); Hs.append(h); Vs.append(v); Ms.append(m)
            As.append(_acc(h,v,m,min(st,2),FLOW[st] if st<3 and fuel[st]>0 else 0.0,vol))
    return (np.array(Ts), np.array(Hs), np.array(Vs), np.array(Ms), np.array(As), ev)

Tf,Hf,Vf,Mf,Af,EV = march("trap")
Td,Hd,Vd,Md,Ad,_   = march("trap", tmax=60.0, every=30)

# ================= estudo de ordem (solução exata conhecida) =================
VE=2700.0; MD=14000.0; M0=2876746.0; TEX=60.0
_aex = lambda t: VE*MD/(M0-MD*t) - G0
_exact = VE*math.log(M0/(M0-MD*TEX)) - G0*TEX

def _err(dt, scheme):
    n=int(round(TEX/dt)); h=TEX/n; v=0.0
    for k in range(n):
        t0=k*h; t1=t0+h
        v += _aex(t0)*h if scheme=="euler" else 0.5*h*(_aex(t0)+_aex(t1))
    return abs(v-_exact)

DTS=[1/60,1/30,1/15,1/8,1/4,1/2]
EE=[_err(d,"euler") for d in DTS]
ET=[_err(d,"trap")  for d in DTS]

# ============================================================
class CenaDiferencasFinitas(Scene):
    def construct(self):
        _title(self, "9 · Diferenças finitas progressivas")
        c = None

        # ---------- ATO 1: contínuo → malha de tempo ----------
        eq = _morph(self, None,
            r"\frac{d\vec R}{dt}=\vec V,\qquad \frac{d\vec V}{dt}=\frac{\vec F(\vec R,\vec V,m)}{m},\qquad \frac{dm}{dt}=-\dot m")
        c = _say(self, c, "Problema de valor inicial: só o estado em t = 0 é conhecido.")

        ax = Line([-5.5,0.35,0],[5.5,0.35,0], color=MUT, stroke_width=2)
        xs = np.linspace(-5.0, 5.0, 9)
        dots = VGroup(*[Dot([x,0.35,0], radius=0.055, color=CYAN) for x in xs])
        names = [r"t_0", r"t_1", r"t_2", r"t_3", r"\cdots", r"t_n", r"t_{n+1}", r"\cdots", r"t_N"]
        labs = VGroup(*[MathTex(names[i], font_size=24, color=MUT).next_to(dots[i], DOWN, buff=0.22)
                        for i in range(9)])
        self.play(Create(ax), run_time=0.7)
        self.play(LaggedStart(*[FadeIn(d) for d in dots], lag_ratio=0.08), Write(labs), run_time=1.4)

        br = Brace(Line([xs[0],0.35,0],[xs[1],0.35,0]), UP, buff=0.12, color=AMBER)
        dtl = MathTex(r"\Delta t=\tfrac{1}{60}\,\mathrm{s}", font_size=26, color=AMBER).next_to(br, UP, buff=0.12)
        self.play(Create(br), Write(dtl), run_time=0.8)

        eq = _morph(self, eq, r"t_n=n\,\Delta t\qquad\Longrightarrow\qquad \vec R_n,\ \vec V_n,\ m_n", fs=34)
        c = _say(self, c, "Marcha progressiva: cada estado novo nasce só do anterior.", AMBER)

        arw = VGroup(*[Arrow([xs[i]+0.10,0.35,0],[xs[i+1]-0.10,0.35,0], buff=0,
                             color=PURPLE, stroke_width=4, max_tip_length_to_length_ratio=0.3)
                       for i in range(8)])
        self.play(LaggedStart(*[GrowArrow(a) for a in arw], lag_ratio=0.12), run_time=1.6)
        self.wait(0.8)
        self.play(FadeOut(VGroup(ax, dots, labs, br, dtl, arw)), FadeOut(c))

        # ---------- ATO 2: diferença progressiva = secante → tangente ----------
        eq = _morph(self, eq,
            r"\frac{dV}{dt}=\lim_{\Delta t\to 0}\frac{V(t+\Delta t)-V(t)}{\Delta t}", fs=34)

        axv = Axes(x_range=[0,4,1], y_range=[0,8,2], x_length=7.4, y_length=3.5, tips=False,
                   axis_config={"color":MUT,"stroke_width":2}).shift(DOWN*0.45 + RIGHT*0.6)
        fv = lambda t: 0.35*t*t + 0.4*t
        curve = axv.plot(fv, x_range=[0,3.95], color=CYAN)
        self.play(Create(axv), Create(curve), run_time=1.1)

        t0 = 1.2; v0 = fv(t0)
        dtv = ValueTracker(2.4)
        P = always_redraw(lambda: Dot(axv.c2p(t0, v0), radius=0.07, color=INK))
        Q = always_redraw(lambda: Dot(axv.c2p(t0+dtv.get_value(), fv(t0+dtv.get_value())),
                                      radius=0.07, color=AMBER))
        def _sec():
            d = dtv.get_value(); s = (fv(t0+d)-v0)/d
            xa = t0 + (0.20 - v0)/s
            xb = min(3.95, t0 + (7.7 - v0)/s)
            return Line(axv.c2p(xa, v0+s*(xa-t0)), axv.c2p(xb, v0+s*(xb-t0)),
                        color=AMBER, stroke_width=3)
        sec = always_redraw(_sec)
        def _span():
            d = dtv.get_value()
            return DoubleArrow(axv.c2p(t0,0), axv.c2p(t0+d,0), buff=0, color=AMBER,
                               stroke_width=3, tip_length=0.12)
        span = always_redraw(_span)
        dtxt = always_redraw(lambda: MathTex(r"\Delta t", font_size=26, color=AMBER)
                             .next_to(_span(), DOWN, buff=0.12))
        vl = always_redraw(lambda: VGroup(
            Line(axv.c2p(t0,0), axv.c2p(t0,v0), color=MUT, stroke_width=1.5).set_stroke(opacity=0.5),
            Line(axv.c2p(t0+dtv.get_value(),0),
                 axv.c2p(t0+dtv.get_value(), fv(t0+dtv.get_value())),
                 color=MUT, stroke_width=1.5).set_stroke(opacity=0.5)))
        self.add(vl, sec, span, P, Q, dtxt)

        eq = _morph(self, eq,
            r"\frac{V_{n+1}-V_n}{\Delta t}\approx a_n\qquad\Longrightarrow\qquad V_{n+1}=V_n+a_n\,\Delta t",
            fs=32, color=AMBER)
        c = _say(self, c, "Diferença progressiva: o declive da secante substitui a derivada.", AMBER)
        self.play(dtv.animate.set_value(0.25), run_time=4.0, rate_func=smooth)

        s_ex = 0.7*t0 + 0.4
        xa = t0 + (0.20-v0)/s_ex; xb = min(3.95, t0 + (7.7-v0)/s_ex)
        tang = DashedLine(axv.c2p(xa, v0+s_ex*(xa-t0)), axv.c2p(xb, v0+s_ex*(xb-t0)),
                          color=GREEN, stroke_width=3, dash_length=0.10)
        self.play(Create(tang), run_time=0.8)
        c = _say(self, c, "Δt finito → erro de truncamento. Euler: O(Δt).", GREEN)
        self.wait(1.2)
        for m in (vl, sec, span, P, Q, dtxt): m.clear_updaters()
        self.play(FadeOut(VGroup(axv, curve, vl, sec, span, P, Q, dtxt, tang)), FadeOut(c))

        # ---------- ATO 3: retângulo (Euler) vs trapézio (notebook) ----------
        eq = _morph(self, eq, r"\Delta V=\int_{t_n}^{t_{n+1}} a(t)\,dt", fs=34)
        axa = Axes(x_range=[0,3,1], y_range=[0,7,2], x_length=7.6, y_length=3.4, tips=False,
                   axis_config={"color":MUT,"stroke_width":2}).shift(DOWN*0.5 + RIGHT*0.5)
        fa = lambda t: 2.0 + 0.9*t + 0.25*t*t
        ca = axa.plot(fa, x_range=[0,2.95], color=CYAN)
        tn, tn1 = 1.15, 1.85
        an, an1 = fa(tn), fa(tn1)
        exact_pts = [axa.c2p(t, fa(t)) for t in np.linspace(tn, tn1, 40)]
        area_true = Polygon(*exact_pts, axa.c2p(tn1,0), axa.c2p(tn,0),
                            fill_color=MUT, fill_opacity=0.22, stroke_width=0)
        rect = Polygon(axa.c2p(tn,0), axa.c2p(tn,an), axa.c2p(tn1,an), axa.c2p(tn1,0),
                       fill_color=RED, fill_opacity=0.42, stroke_color=RED, stroke_width=2)
        trap = Polygon(axa.c2p(tn,0), axa.c2p(tn,an), axa.c2p(tn1,an1), axa.c2p(tn1,0),
                       fill_color=GREEN, fill_opacity=0.42, stroke_color=GREEN, stroke_width=2)
        tri = Polygon(axa.c2p(tn,an), axa.c2p(tn1,an), axa.c2p(tn1,an1),
                      fill_color=AMBER, fill_opacity=0.55, stroke_width=0)
        vln = DashedLine(axa.c2p(tn,0), axa.c2p(tn,6.6), color=MUT, stroke_width=1.5, dash_length=0.08)
        vln1 = DashedLine(axa.c2p(tn1,0), axa.c2p(tn1,6.6), color=MUT, stroke_width=1.5, dash_length=0.08)
        nl = MathTex(r"t_n", font_size=24, color=MUT).next_to(axa.c2p(tn,0), DOWN, buff=0.18)
        nl1 = MathTex(r"t_{n+1}", font_size=24, color=MUT).next_to(axa.c2p(tn1,0), DOWN, buff=0.18)

        self.play(Create(axa), Create(ca), Create(vln), Create(vln1), Write(nl), Write(nl1), run_time=1.1)
        self.play(FadeIn(area_true), run_time=0.5)
        self.play(FadeIn(rect), run_time=0.7)
        eq = _morph(self, eq, r"\Delta V\approx a_n\,\Delta t", fs=34, color=RED)
        c = _say(self, c, "Euler progressivo = retângulo. Falta a fatia acima.", RED)
        self.wait(0.9)

        self.play(FadeIn(tri), run_time=0.6)
        self.play(ReplacementTransform(rect, trap), FadeOut(tri), run_time=1.1)
        eq = _morph(self, eq, r"\Delta V\approx \frac{\Delta t}{2}\left(a_n+a_{n+1}\right)", fs=34, color=GREEN)
        c = _say(self, c, "O notebook usa a média das duas pontas: trapézio, O(Δt²).", GREEN)
        self.wait(1.1)
        self.play(FadeOut(VGroup(axa, ca, area_true, trap, vln, vln1, nl, nl1)), FadeOut(c))

        # ---------- ATO 4: o laço do notebook (CORRIGIDO) ----------
        boxes = [
            _box(r"\vec R_n,\vec V_n,m_n", "estado", CYAN).move_to([-4.6, 0.55, 0]),
            _box(r"\vec F", "resultante", PURPLE).move_to([-1.55, 0.55, 0]),
            _box(r"a_n=\vec F/m_n", "aceleração", AMBER).move_to([1.6, 0.55, 0]),
            _box(r"\vec R_{n+1},\vec V_{n+1},m_{n+1}", "novo estado", GREEN).move_to([4.85, 0.55, 0]),
        ]
        ar = VGroup(*[Arrow(boxes[i].get_right()+RIGHT*0.06, boxes[i+1].get_left()+LEFT*0.06,
                            buff=0, color=MUT, stroke_width=3, max_tip_length_to_length_ratio=0.28)
                      for i in range(3)])
        loop = ArcBetweenPoints(boxes[3].get_bottom()+DOWN*0.05, boxes[0].get_bottom()+DOWN*0.05,
                                angle=-1.5, color=PINK, stroke_width=3)
        loop_tip = Arrow(loop.point_from_proportion(0.93), loop.get_end(), buff=0,
                         color=PINK, stroke_width=3, max_tip_length_to_length_ratio=0.5)
        loop_l = MathTex(r"n\leftarrow n+1", font_size=26, color=PINK).move_to([0,-2.05,0])

        eq = _morph(self, eq, r"n=0,\,1,\,2,\,\dots", fs=32, at=UP*2.5)
        c = _say(self, c, "Laço explícito: cada passo usa apenas o estado anterior.", MUT)
        self.play(FadeIn(boxes[0]), run_time=0.5)
        for i in range(3):
            self.play(GrowArrow(ar[i]), FadeIn(boxes[i+1]), run_time=0.6)
        self.play(Create(loop), FadeIn(loop_tip), Write(loop_l), run_time=1.0)

        upd = VGroup(
            MathTex(r"\vec V_{n+1}=\vec V_n+\tfrac{\Delta t}{2}(a_n+a_{n+1})", font_size=28, color=GREEN),
            MathTex(r"\vec R_{n+1}=\vec R_n+\tfrac{\Delta t}{2}(\vec V_n+\vec V_{n+1})", font_size=28, color=CYAN),
            MathTex(r"m_{n+1}=m_n-\tfrac{\Delta t}{2}(\dot m_n+\dot m_{n+1})", font_size=28, color=AMBER),
        ).arrange(DOWN, buff=0.20, aligned_edge=LEFT).move_to([0,-1.05,0])
        self.play(FadeOut(loop_l), FadeOut(loop), FadeOut(loop_tip), run_time=0.4)
        self.play(LaggedStart(*[Write(u) for u in upd], lag_ratio=0.25), run_time=2.0)
        c = _say(self, c, "a_{n+1} exige uma previsão de Euler antes da média.", MUT)
        self.wait(1.3)
        self.play(FadeOut(VGroup(*boxes, ar, upd)), FadeOut(c))

        # ---------- ATO 5: marcha ao vivo com o foguete ----------
        eq = _morph(self, eq, r"\Delta t=\tfrac{1}{60}\,\mathrm{s}\quad\Rightarrow\quad 3600\ \mathrm{passos\ em}\ 60\,\mathrm{s}",
                    fs=30, at=UP*2.5)

        y0, y1 = -2.55, 1.55
        alt = Line([-5.5,y0,0],[-5.5,y1,0], color=MUT, stroke_width=2)
        ticks = VGroup(); tlabs = VGroup()
        for km in range(0, 11, 2):
            yy = y0 + (km/10.0)*(y1-y0)
            ticks.add(Line([-5.62,yy,0],[-5.38,yy,0], color=MUT, stroke_width=2))
            tlabs.add(Text(f"{km}", font_size=14, color=MUT).next_to([-5.62,yy,0], LEFT, buff=0.12))
        altl = Text("km", font_size=15, color=MUT).next_to(alt, UP, buff=0.12)

        axh = Axes(x_range=[0,60,15], y_range=[0,10,2], x_length=6.6, y_length=3.6, tips=False,
                   axis_config={"color":MUT,"stroke_width":2,"include_numbers":True,
                                "font_size":16}).shift(RIGHT*2.0 + DOWN*0.45)
        hl = axh.get_axis_labels(MathTex("t", font_size=24), MathTex("h", font_size=24))
        pts = [axh.c2p(tt, hh/1000.0) for tt, hh in zip(Td, Hd)]
        u = ValueTracker(0.0)
        frac = np.linspace(0, 1, len(Hd))
        trace = always_redraw(lambda: _poly(
            pts[:max(2, int(u.get_value()*(len(pts)-1))+1)], CYAN, 3))
        rk = _rocket_icon(0.95)
        def _place(mob, dt=0):                      # updater recebe (mob, dt)
            hh = float(np.interp(u.get_value(), frac, Hd))
            mob.move_to([-5.5, y0 + (min(hh,10000)/10000.0)*(y1-y0) + mob.height/2, 0])
        _place(rk); rk.add_updater(_place)
        trail = always_redraw(lambda: VGroup(*[
            Dot([-5.5, y0 + (min(Hd[j],10000)/10000.0)*(y1-y0), 0], radius=0.035, color=PURPLE)
            for j in range(0, int(u.get_value()*(len(Hd)-1))+1, 6)]))

        led = VGroup(
            Text("n", font_size=15, color=MUT),
            Text("t (s)", font_size=15, color=MUT),
            Text("v (m/s)", font_size=15, color=MUT),
            Text("h (m)", font_size=15, color=MUT),
        ).arrange(DOWN, buff=0.26, aligned_edge=LEFT).move_to([-3.35, 0.55, 0])
        nums = VGroup()
        for i, arr in enumerate([None, Td, Vd, Hd]):
            dn = DecimalNumber(0, num_decimal_places=0 if i in (0,3) else 1,
                               font_size=24, color=CYAN)
            dn.next_to(led[i], RIGHT, buff=0.30)
            src = np.arange(3601) if i == 0 else arr
            fr = np.linspace(0, 1, len(src))
            dn.add_updater(lambda m, dt, s=src, f=fr:
                           m.set_value(float(np.interp(u.get_value(), f, s))))
            nums.add(dn)
        lb = RoundedRectangle(corner_radius=0.12, width=2.5, height=2.0, stroke_color="#31456b",
                              stroke_width=1.2, fill_color="#0a1120", fill_opacity=0.85)
        lb.move_to(Group(led, nums).get_center())

        self.play(Create(alt), Write(ticks), Write(tlabs), Write(altl),
                  Create(axh), Write(hl), FadeIn(lb), Write(led), FadeIn(nums), run_time=1.2)
        self.add(trail, trace, rk)
        c = _say(self, c, "Cada ponto da curva é um passo finito do laço.", CYAN)
        self.play(u.animate.set_value(1.0), run_time=9.0, rate_func=linear)
        self.wait(0.8)
        rk.clear_updaters()
        for dn in nums: dn.clear_updaters()
        trace.clear_updaters(); trail.clear_updaters()
        self.play(FadeOut(Group(alt, ticks, tlabs, altl, axh, hl, lb, led, nums,
                                trace, trail, rk)), FadeOut(c))

        # ---------- ATO 6: ordem de convergência medida ----------
        eq = _morph(self, eq, r"\left|V_{\mathrm{num}}-V_{\mathrm{exato}}\right|\ \mathrm{em}\ t=60\,\mathrm{s}",
                    fs=30, at=UP*2.5)
        axe = Axes(x_range=[-2.0,-0.1,0.5], y_range=[-6.5,0.6,1], x_length=7.6, y_length=3.7,
                   tips=False, axis_config={"color":MUT,"stroke_width":2}).shift(DOWN*0.45+RIGHT*0.4)
        xe = [math.log10(d) for d in DTS]
        ye = [math.log10(v) for v in EE]
        yt = [math.log10(v) for v in ET]
        grid = VGroup(); xlab = VGroup(); ylab = VGroup()
        for k in (-6,-5,-4,-3,-2,-1,0):
            yy = axe.c2p(-2.0, k)[1]
            grid.add(DashedLine(axe.c2p(-2.0,k), axe.c2p(-0.1,k), dash_length=0.06,
                                stroke_width=1, color=MUT, stroke_opacity=0.20))
            ylab.add(MathTex(f"10^{{{k}}}", font_size=20, color=MUT)
                     .next_to(axe.c2p(-2.0,k), LEFT, buff=0.14))
        for d in DTS:
            xx = axe.c2p(math.log10(d), -6.5)[0]
            t = Text(f"1/{int(round(1/d))}", font_size=15, color=MUT)
            t.next_to([xx, axe.c2p(0,-6.5)[1], 0], DOWN, buff=0.14)
            xlab.add(t)
        exl = MathTex(r"\Delta t", font_size=24, color=MUT).next_to(axe, DOWN, buff=0.55)
        eyl = MathTex(r"|\mathrm{erro}|", font_size=22, color=MUT).next_to(axe, LEFT, buff=0.75)

        pE = _poly([axe.c2p(a,b) for a,b in zip(xe,ye)], RED, 3)
        pT = _poly([axe.c2p(a,b) for a,b in zip(xe,yt)], GREEN, 3)
        dE = VGroup(*[Dot(axe.c2p(a,b), radius=0.055, color=RED) for a,b in zip(xe,ye)])
        dT = VGroup(*[Dot(axe.c2p(a,b), radius=0.055, color=GREEN) for a,b in zip(xe,yt)])
        rE = DashedLine(axe.c2p(-1.95, ye[0]+1*(-1.95-xe[0])), axe.c2p(-0.15, ye[0]+1*(-0.15-xe[0])),
                        color=RED, stroke_width=1.6, dash_length=0.09).set_stroke(opacity=0.55)
        rT = DashedLine(axe.c2p(-1.95, yt[0]+2*(-1.95-xe[0])), axe.c2p(-0.15, yt[0]+2*(-0.15-xe[0])),
                        color=GREEN, stroke_width=1.6, dash_length=0.09).set_stroke(opacity=0.55)
        lE = MathTex(r"\mathcal{O}(\Delta t)", font_size=26, color=RED).move_to(axe.c2p(-0.55,-0.35))
        lT = MathTex(r"\mathcal{O}(\Delta t^2)", font_size=26, color=GREEN).move_to(axe.c2p(-0.60,-3.0))

        self.play(Create(axe), Write(grid), Write(ylab), Write(xlab), Write(exl), Write(eyl), run_time=1.2)
        self.play(Create(pE), FadeIn(dE), Create(rE), Write(lE), run_time=1.4)
        self.play(Create(pT), FadeIn(dT), Create(rT), Write(lT), run_time=1.4)
        c = _say(self, c, "Inclinação 1 = Euler · inclinação 2 = trapézio do notebook.", GREEN)
        self.wait(1.6)
        self.play(FadeOut(VGroup(axe, grid, ylab, xlab, exl, eyl, pE, pT, dE, dT, rE, rT, lE, lT)), FadeOut(c))

        # ---------- ATO 7: o voo inteiro + saltos de estágio ----------
        eq = _morph(self, eq,
            r"c_g\ge\textstyle\sum m_c\ \Rightarrow\ m\!\leftarrow\! m-m_s,\quad \vec V\!\leftarrow\!\vec V+\Delta\vec v,\quad A\!\leftarrow\! A'",
            fs=28, at=UP*2.5)

        axm = Axes(x_range=[0,780,180], y_range=[0,3000,1000], x_length=6.0, y_length=2.3,
                   tips=False, axis_config={"color":MUT,"stroke_width":2,"font_size":14,
                                            "include_numbers":True}).shift(UP*0.55+RIGHT*0.4)
        axa2 = Axes(x_range=[0,780,180], y_range=[0,160,40], x_length=6.0, y_length=2.3,
                    tips=False, axis_config={"color":MUT,"stroke_width":2,"font_size":14,
                                             "include_numbers":True}).shift(DOWN*1.85+RIGHT*0.4)
        lm = MathTex(r"m\ [\mathrm{t}]", font_size=22, color=AMBER).next_to(axm, LEFT, buff=0.30)
        la = MathTex(r"a\ [\mathrm{m/s^2}]", font_size=22, color=PINK).next_to(axa2, LEFT, buff=0.30)
        cm = _poly([axm.c2p(t, m/1000.0) for t,m in zip(Tf,Mf)], AMBER, 3)
        ca2 = _poly([axa2.c2p(t, min(a,158)) for t,a in zip(Tf,Af)], PINK, 3)

        self.play(Create(axm), Create(axa2), Write(lm), Write(la), run_time=1.0)
        self.play(Create(cm), Create(ca2), run_time=2.2)

        marks = VGroup(); mlabs = VGroup()
        nomes_ev = ["S1→S2", "S2→S3", "MECO"]
        for i,(tt, stg, dv) in enumerate(EV[:3]):
            for A in (axm, axa2):
                yr = A.y_range
                marks.add(DashedLine(A.c2p(tt, yr[0]), A.c2p(tt, yr[1]), color=PURPLE,
                                     stroke_width=1.8, dash_length=0.08))
            mlabs.add(Text(nomes_ev[i], font_size=14, color=PURPLE)
                      .next_to(axm.c2p(tt, axm.y_range[1]), UP, buff=0.10))
        self.play(LaggedStart(*[Create(m) for m in marks], lag_ratio=0.1),
                  Write(mlabs), run_time=1.4)

        dvl = MathTex(r"\Delta v=\sqrt{\frac{m_s E}{M_1(M_1+m_s)}}", font_size=26, color=CYAN)
        dvl.to_corner(DL, buff=0.55).shift(UP*0.15)
        self.play(Write(dvl), run_time=0.8)
        c = _say(self, c, "Eventos discretos entram como saltos no estado — o laço continua.", PURPLE)
        self.wait(2.5)

<string>:104: RuntimeWarning: divide by zero encountered in divide


TypeError: only length-1 arrays can be converted to Python scalars

In [29]:
import os
import numpy as np
from manim import *
from IPython.display import Video, display


class EvolucaoTemporalFoguete(Scene):
    def criar_foguete(self, escala=0.55):
        asset_path = "media/assets/foguete.png"
        if os.path.exists(asset_path):
            foguete = ImageMobject(asset_path)
            foguete.scale_to_fit_height(2.4 * escala)
            return foguete

        # Fallback vetorial caso media/assets/foguete.png não esteja no diretório atual
        corpo = RoundedRectangle(
            corner_radius=0.12, width=0.55, height=1.8,
            fill_color="#E6EDF3", fill_opacity=1, stroke_color="#58A6FF", stroke_width=2
        )
        coifa = Polygon(
            [-0.275, 0.9, 0], [0.275, 0.9, 0], [0, 1.45, 0],
            fill_color="#F85149", fill_opacity=1, stroke_width=0
        )
        aleta_esq = Polygon(
            [-0.275, -0.9, 0], [-0.55, -1.1, 0], [-0.275, -0.35, 0],
            fill_color="#58A6FF", fill_opacity=1, stroke_width=0
        )
        aleta_dir = Polygon(
            [0.275, -0.9, 0], [0.55, -1.1, 0], [0.275, -0.35, 0],
            fill_color="#58A6FF", fill_opacity=1, stroke_width=0
        )
        chama = Polygon(
            [-0.2, -0.9, 0], [0.2, -0.9, 0], [0, -1.45, 0],
            fill_color="#FFA657", fill_opacity=0.95, stroke_width=0
        )
        return VGroup(chama, aleta_esq, aleta_dir, corpo, coifa).scale(escala)

    def construct(self):
        # Fundo completamente preto
        self.camera.background_color = "#000000"

        # ======================================================================
        # TÍTULO PRINCIPAL
        # ======================================================================
        titulo = Text(
            "Evolução Temporal — Método de Diferenças Finitas",
            font_size=30, weight=BOLD, color="#58A6FF"
        ).to_edge(UP, buff=0.35)
        subtitulo = MathTex(
            r"\Delta t = \frac{1}{60}\,\mathrm{s}", font_size=28, color="#3FB950"
        ).next_to(titulo, DOWN, buff=0.15)

        self.play(Write(titulo), FadeIn(subtitulo, shift=DOWN * 0.15))
        self.wait(0.6)

        # ======================================================================
        # ATO 1: MALHA DISCRETA E APROXIMAÇÃO DAS DERIVADAS
        # ======================================================================
        linha_tempo = Line(LEFT * 5.0, RIGHT * 5.0, color="#8B949E", stroke_width=3).shift(UP * 1.55)
        xs = [-4.0, -1.33, 1.33, 4.0]
        labels_t = [r"t_{n-2}", r"t_{n-1}", r"t_n", r"t_{n+1}"]
        pontos_t = VGroup()

        for x, lbl in zip(xs, labels_t):
            cor = "#3FB950" if lbl == r"t_n" else ("#58A6FF" if lbl == r"t_{n-1}" else "#8B949E")
            ponto = Dot(point=[x, 1.55, 0], radius=0.09, color=cor)
            txt = MathTex(lbl, font_size=26, color=cor).next_to(ponto, UP, buff=0.18)
            pontos_t.add(VGroup(ponto, txt))

        seg_dt = Line([-1.33, 1.55, 0], [1.33, 1.55, 0], color="#3FB950", stroke_width=6)
        chave_dt = Brace(seg_dt, DOWN, color="#3FB950", buff=0.12)
        txt_dt = chave_dt.get_tex(r"\Delta t = \frac{1}{60}\,\mathrm{s}").scale(0.76).set_color("#3FB950")

        self.play(Create(linha_tempo), FadeIn(pontos_t, lag_ratio=0.15))
        self.play(Create(seg_dt), GrowFromCenter(chave_dt), Write(txt_dt))
        self.wait(0.5)

        # Card com margem superior ampliada em relação à chave \Delta t
        eq_derivadas = MathTex(
            r"\frac{d\vec{v}}{dt} \approx \frac{\vec{v}_n - \vec{v}_{n-1}}{\Delta t}, \qquad"
            r"\frac{d\vec{r}}{dt} \approx \frac{\vec{r}_n - \vec{r}_{n-1}}{\Delta t}, \qquad"
            r"\frac{dm}{dt} \approx \frac{m_n - m_{n-1}}{\Delta t}",
            font_size=27, color="#E6EDF3"
        ).shift(DOWN * 1.95)

        caixa_deriv = SurroundingRectangle(
            eq_derivadas, corner_radius=0.15, buff=0.32,
            color="#30363D", fill_color="#0D1117", fill_opacity=0.95
        )
        rotulo_deriv = Text(
            "1. Discretização das Taxas de Variação", font_size=20, color="#58A6FF", weight=BOLD
        ).next_to(caixa_deriv, UP, aligned_edge=LEFT, buff=0.22)

        self.play(FadeIn(caixa_deriv), FadeIn(rotulo_deriv), Write(eq_derivadas))
        self.wait(1.6)

        self.play(
            FadeOut(linha_tempo), FadeOut(pontos_t), FadeOut(seg_dt),
            FadeOut(chave_dt), FadeOut(txt_dt), FadeOut(caixa_deriv),
            FadeOut(rotulo_deriv), FadeOut(eq_derivadas), FadeOut(subtitulo)
        )

        # ======================================================================
        # ATO 2: BALANÇO DE FORÇAS E ACELERAÇÃO INSTANTÂNEA EM t_{n-1}
        # ======================================================================
        foguete = self.criar_foguete(escala=0.85).move_to(LEFT * 4.2 + UP * 0.15)

        # Vetores de força
        seta_em = Arrow(foguete.get_top(), foguete.get_top() + UP * 1.35, buff=0.05, color="#3FB950", stroke_width=5)
        lbl_em = MathTex(r"\vec{E}_m", font_size=24, color="#3FB950").next_to(seta_em, RIGHT, buff=0.12)

        seta_ea = Arrow(foguete.get_top() + LEFT * 0.25, foguete.get_top() + LEFT * 0.25 + UP * 0.85, buff=0.05, color="#3FB950", stroke_width=4)
        lbl_ea = MathTex(r"\vec{E}_a", font_size=24, color="#3FB950").next_to(seta_ea, LEFT, buff=0.12)

        seta_p = Arrow(foguete.get_bottom(), foguete.get_bottom() + DOWN * 1.15, buff=0.05, color="#F85149", stroke_width=5)
        lbl_p = MathTex(r"\vec{P}", font_size=24, color="#F85149").next_to(seta_p, RIGHT, buff=0.12)

        seta_fd = Arrow(foguete.get_bottom() + LEFT * 0.25, foguete.get_bottom() + LEFT * 0.25 + DOWN * 0.75, buff=0.05, color="#F85149", stroke_width=4)
        lbl_fd = MathTex(r"\vec{F}_D", font_size=24, color="#F85149").next_to(seta_fd, LEFT, buff=0.12)

        # Posiciona o rótulo abaixo da ponta da seta mais longa (\vec{P}), centralizado no foguete
        estado_ant = MathTex(
            r"t_{n-1}:\; (\vec{r}_{n-1},\, \vec{v}_{n-1},\, m_{n-1})",
            font_size=24, color="#58A6FF"
        ).next_to(seta_p, DOWN, buff=0.28).set_x(foguete.get_x())

        forcas_grp = VGroup(seta_em, lbl_em, seta_ea, lbl_ea, seta_p, lbl_p, seta_fd, lbl_fd)

        # Equação da aceleração resultante
        titulo_acel = Text("2. Resultante e Aceleração no Passo n", font_size=22, color="#58A6FF", weight=BOLD)
        eq_acel = MathTex(
            r"\vec{a}_n = \frac{\vec{F}_R(\vec{r}_{n-1},\, \vec{v}_{n-1},\, m_{n-1})}{m_{n-1}}",
            font_size=28, color="#E6EDF3"
        )
        eq_acel_det = MathTex(
            r"\vec{a}_n = \frac{\vec{P} + \vec{E}_a + \vec{F}_D + \vec{E}_m}{m_{n-1}}",
            font_size=28, color="#D2A8FF"
        )
        bloco_acel = VGroup(titulo_acel, eq_acel, eq_acel_det).arrange(DOWN, buff=0.35).move_to(RIGHT * 2.1 + UP * 0.1)
        caixa_acel = SurroundingRectangle(
            bloco_acel, corner_radius=0.15, buff=0.32,
            color="#30363D", fill_color="#0D1117", fill_opacity=0.95
        )

        self.play(FadeIn(foguete), Write(estado_ant))
        self.play(GrowArrow(seta_em), FadeIn(lbl_em), GrowArrow(seta_ea), FadeIn(lbl_ea))
        self.play(GrowArrow(seta_p), FadeIn(lbl_p), GrowArrow(seta_fd), FadeIn(lbl_fd))
        self.play(FadeIn(caixa_acel), Write(titulo_acel), Write(eq_acel))
        self.wait(0.5)
        self.play(Write(eq_acel_det))
        self.wait(1.4)

        # ======================================================================
        # ATO 3: INTEGRAÇÃO TEMPORAL (FOGUETE INCLINADO + PARTÍCULAS + SEM SOBREPOSIÇÃO)
        # ======================================================================
        self.play(
            FadeOut(forcas_grp), FadeOut(caixa_acel), FadeOut(bloco_acel)
        )

        # Coordenadas calibradas para manter margem segura de todos os textos
        p0 = np.array([-5.0, -1.35, 0.0])
        p1 = np.array([-3.3, 0.55, 0.0])

        ponto_ini = Dot(p0, color="#58A6FF", radius=0.085)
        ponto_fim = Dot(p1, color="#3FB950", radius=0.085)

        traj_curve = ArcBetweenPoints(p0, p1, angle=-0.28)
        traj_arco = DashedVMobject(
            traj_curve.copy().set_color("#8B949E").set_stroke(width=2.5),
            num_dashes=14
        )
        seta_dr = Arrow(
            p0, p1, buff=0.08, color="#3FB950", stroke_width=3.2
        )

        # Rótulos posicionados fora da trajetória e do envelope do foguete
        lbl_dr = MathTex(r"\Delta \vec{r}", font_size=24, color="#3FB950").move_to(
            np.array([-3.45, -0.65, 0.0])
        )
        estado_novo = MathTex(
            r"t_n:\; (\vec{r}_n,\, \vec{v}_n,\, m_n)",
            font_size=23, color="#3FB950"
        ).move_to(np.array([-3.85, 1.72, 0.0]))

        # Função que calcula o ângulo tangente da curva em relação à vertical (UP)
        def angulo_tangente(alpha):
            eps = 1e-3
            a0 = max(0.0, alpha - eps)
            a1 = min(1.0, alpha + eps)
            tang = traj_curve.point_from_proportion(a1) - traj_curve.point_from_proportion(a0)
            return np.arctan2(tang[1], tang[0]) - PI / 2

        ang_inicial = angulo_tangente(0.0)
        foguete_voo_base = self.criar_foguete(escala=0.48)

        # Posiciona e inclina o foguete no ponto inicial t_{n-1}
        self.play(
            foguete.animate.scale_to_fit_height(foguete_voo_base.height).rotate(ang_inicial).move_to(p0),
            estado_ant.animate.scale(0.95).move_to(np.array([-3.95, -2.55, 0.0])),
            FadeIn(ponto_ini),
            run_time=0.8
        )

        # Sistema de partículas de exaustão ao longo do voo
        rng = np.random.default_rng(42)
        n_particulas = 34
        dados_particulas = []
        cores_fogo = ["#FFF3B0", "#FFD166", "#FFA657", "#F85149", "#8B949E"]

        for i in range(n_particulas):
            tau_nasc = (i / n_particulas) * 0.96
            vida = rng.uniform(0.24, 0.44)
            espalhamento = rng.uniform(-0.18, 0.18)
            velocidade_escape = rng.uniform(0.32, 0.58)
            raio_ini = rng.uniform(0.032, 0.058)
            cor_idx = int(rng.integers(0, 3))
            dados_particulas.append((tau_nasc, vida, espalhamento, velocidade_escape, raio_ini, cor_idx))

        progresso = ValueTracker(0.0)
        meia_altura = foguete_voo_base.height * 0.50

        def atualizar_foguete(mob):
            alpha = progresso.get_value()
            pos = traj_curve.point_from_proportion(alpha)
            ang = angulo_tangente(alpha)
            mob.become(foguete_voo_base.copy().rotate(ang).move_to(pos))

        def desenhar_particulas():
            alpha = progresso.get_value()
            grupo = VGroup()
            for tau, vida, spread, vel, r0, c_idx in dados_particulas:
                if alpha < tau:
                    continue
                idade = (alpha - tau) / vida
                if idade > 1.0:
                    if alpha >= 0.99 and tau > 0.78:
                        idade = min(0.85, (alpha - tau) / 0.35)
                    else:
                        continue

                pos_fog = traj_curve.point_from_proportion(tau)
                ang = angulo_tangente(tau)
                dir_frente = np.array([-np.sin(ang), np.cos(ang), 0.0])
                dir_perp = np.array([dir_frente[1], -dir_frente[0], 0.0])

                bocal = pos_fog - meia_altura * dir_frente
                desloc = (-vel * idade) * dir_frente + (spread * idade) * dir_perp
                pos_p = bocal + desloc

                idx_atual = min(len(cores_fogo) - 1, c_idx + int(idade * 2.5))
                opacidade = max(0.0, 0.9 * (1.0 - idade) ** 1.3)
                raio = r0 * (1.0 + 0.65 * idade)

                part = Dot(
                    point=pos_p, radius=raio,
                    color=cores_fogo[idx_atual], fill_opacity=opacidade
                )
                grupo.add(part)
            return grupo

        particulas_layer = always_redraw(desenhar_particulas)
        self.add(particulas_layer)
        self.bring_to_front(foguete)
        foguete.add_updater(atualizar_foguete)

        self.play(
            Create(traj_arco),
            progresso.animate.set_value(1.0),
            FadeIn(ponto_fim),
            GrowArrow(seta_dr),
            FadeIn(lbl_dr),
            Write(estado_novo),
            run_time=1.5,
            rate_func=smooth
        )
        foguete.remove_updater(atualizar_foguete)

        # Equações trapezoidais na direita
        titulo_trap = Text("3. Passo de Integração (Regra Trapezoidal)", font_size=21, color="#58A6FF", weight=BOLD)
        eq_v = MathTex(
            r"\vec{v}_n = \vec{v}_{n-1} + \frac{\Delta t}{2}\left(\vec{a}_{n-1} + \vec{a}_n\right)",
            font_size=27, color="#3FB950"
        )
        eq_r = MathTex(
            r"\vec{r}_n = \vec{r}_{n-1} + \frac{\Delta t}{2}\left(\vec{v}_{n-1} + \vec{v}_n\right)",
            font_size=27, color="#58A6FF"
        )
        eq_m = MathTex(
            r"m_n = m_{n-1} - \frac{\Delta t}{2}\left(\dot{m}_{n-1} + \dot{m}_n\right)",
            font_size=27, color="#FFA657"
        )

        bloco_trap = VGroup(titulo_trap, eq_v, eq_r, eq_m).arrange(DOWN, aligned_edge=LEFT, buff=0.35).move_to(RIGHT * 2.1 + DOWN * 0.1)
        caixa_trap = SurroundingRectangle(
            bloco_trap, corner_radius=0.15, buff=0.3,
            color="#30363D", fill_color="#0D1117", fill_opacity=0.95
        )

        self.play(FadeIn(caixa_trap), Write(titulo_trap))
        for eq in [eq_v, eq_r, eq_m]:
            self.play(Write(eq), run_time=0.75)
            self.play(Indicate(eq, color="#E6EDF3", scale_factor=1.03), run_time=0.45)

        self.wait(1.4)

        # ======================================================================
        # ATO 4: TRANSIÇÃO DISCRETA — EJEÇÃO DE ESTÁGIOS
        # ======================================================================
        particulas_layer.clear_updaters()
        self.play(
            FadeOut(foguete), FadeOut(particulas_layer),
            FadeOut(ponto_ini), FadeOut(ponto_fim),
            FadeOut(traj_arco), FadeOut(seta_dr), FadeOut(lbl_dr),
            FadeOut(estado_ant), FadeOut(estado_novo),
            FadeOut(caixa_trap), FadeOut(bloco_trap)
        )

        # Representação esquemática dos estágios M1 (superior) e M2 (ejetado)
        estagio_sup = RoundedRectangle(
            corner_radius=0.1, width=0.65, height=1.25,
            fill_color="#E6EDF3", fill_opacity=1, stroke_color="#58A6FF", stroke_width=2
        )
        coifa_sup = Polygon(
            [-0.325, 0.625, 0], [0.325, 0.625, 0], [0, 1.25, 0],
            fill_color="#F85149", fill_opacity=1, stroke_width=0
        )
        lbl_m1 = MathTex(r"M_1", font_size=24, color="#000000").move_to(estagio_sup.get_center())
        grupo_m1 = VGroup(estagio_sup, coifa_sup, lbl_m1).move_to(LEFT * 4.0 + UP * 0.45)

        estagio_inf = RoundedRectangle(
            corner_radius=0.06, width=0.72, height=1.1,
            fill_color="#FFA657", fill_opacity=0.95, stroke_color="#F85149", stroke_width=2
        )
        lbl_m2 = MathTex(r"M_2", font_size=24, color="#000000").move_to(estagio_inf.get_center())
        grupo_m2 = VGroup(estagio_inf, lbl_m2).next_to(grupo_m1, DOWN, buff=0.04)

        titulo_ej = Text("4. Evento Discreto: Ejeção de Estágios", font_size=22, color="#F85149", weight=BOLD)
        cond_ej = MathTex(
            r"\text{Condição: }\; m_{\mathrm{queimado}} \ge m_{c,\mathrm{est\acute{a}gio}}",
            font_size=26, color="#FFA657"
        )
        eq_ej_m = MathTex(
            r"m_n \leftarrow m_n - M_2",
            font_size=28, color="#E6EDF3"
        )
        eq_ej_v = MathTex(
            r"\vec{v}_n \leftarrow \vec{v}_n + \sqrt{\frac{M_2\,E}{M_1(M_1 + M_2)}}\,\hat{u}",
            font_size=28, color="#3FB950"
        )

        bloco_ej = VGroup(titulo_ej, cond_ej, eq_ej_m, eq_ej_v).arrange(DOWN, aligned_edge=LEFT, buff=0.35).move_to(RIGHT * 1.9 + DOWN * 0.1)
        caixa_ej = SurroundingRectangle(
            bloco_ej, corner_radius=0.15, buff=0.3,
            color="#30363D", fill_color="#0D1117", fill_opacity=0.95
        )

        self.play(FadeIn(grupo_m1), FadeIn(grupo_m2), FadeIn(caixa_ej), Write(titulo_ej), Write(cond_ej))
        self.wait(0.5)

        # Animação da separação + impulso \Delta v
        seta_dv = Arrow(
            grupo_m1.get_top() + UP * 0.1, grupo_m1.get_top() + UP * 1.15,
            buff=0, color="#3FB950", stroke_width=6
        )
        lbl_dv = MathTex(r"+\Delta \vec{v}", font_size=26, color="#3FB950").next_to(seta_dv, RIGHT, buff=0.15)

        self.play(
            Write(eq_ej_m),
            Write(eq_ej_v),
            grupo_m2.animate.shift(DOWN * 1.5 + LEFT * 0.35).rotate(0.25).set_opacity(0.35),
            grupo_m1.animate.shift(UP * 0.65),
            run_time=1.3
        )
        self.play(GrowArrow(seta_dv), FadeIn(lbl_dv))
        self.wait(2.0)


# ==============================================================================
# RENDERIZAÇÃO E EXIBIÇÃO NO JUPYTER
# ==============================================================================
with tempconfig({"quality": "medium_quality", "preview": False, "verbosity": "WARNING"}):
    cena = EvolucaoTemporalFoguete()
    cena.render()
    display(Video(cena.renderer.file_writer.movie_file_path, embed=True))

In [ ]:
# Os vídeos já foram salvos em media/videos/<nome_da_cena>/720p30/.
# Para 1080p60, troque -qm por -qh em cada célula e re-execute.
#
# Alternativa fora do notebook: salve as classes das cenas em um arquivo
# pen_derivacoes.py (com `from manim import *` e os helpers) e rode:
#   !manim -qh pen_derivacoes.py -a
#
# Listar o que foi gerado:
import pathlib
for p in sorted(pathlib.Path("media/videos").rglob("*.mp4")):
    print(p)